# Teste MT

In [157]:
import pandas as pd
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [18]:
funcoes = {
    
    'MT': rectify_MT,
}

lista_estados = ['MT']

tabela_ids = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv')
tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')


In [72]:

estado = 'DF'
  
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

#df_ids = funcoes[estado](path)
    
#create_df_estacao(estado,df_ids)

In [52]:
dict_pols_stat = defaultdict(list)

files = os.listdir(path)

print(files)

for item in files:
    
    estacao = " ".join(item.split('-')[1].split('.')[0].split('_')[0:2])

    print(estacao)

    df = pd.read_excel(path+item)

    df = df.drop(columns=['Nome da estação'])

    lista_pols = set(df['Poluente'])

    for pol in lista_pols:

        df_pol = df[df["Poluente"] == pol]
        
        if pol in ['no2','so2','o3']:

            df_pol = ppb_to_ug(df_pol,pol)

        df_pol_hora = df_pol.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"])
        
        df_pol_hora = df_pol_hora.filter(lambda g: len(g) >= 9)
        
        df_pol = (
            df_pol_hora.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"], as_index=False)
                  .agg({"Valor": "mean"})
        )

        df_pol['QAQC_INTERNO'] = None

        df_pol = df_pol.rename(columns={'Ano':'ANO',
                                        'Mes':'MES',
                                        'Dia':'DIA',
                                        'Hora':'HORA',
                                        'Unidade':'UNIDADE',
                                        'Valor':'VALOR'})

        for col in ["ANO", "MES", "DIA", "HORA"]:
            df_pol[col] = pd.to_numeric(df_pol[col], errors="coerce").astype("Int64")
        
        dict_pols_stat[estacao+'_'+pol].append(df_pol)



['dados_monitoramento-Sema.xlsx', 'dados_monitoramento-BEA_CBA_24-25.xlsx', 'dados_monitoramento-BEA_CBA_22-23.xlsx', 'dados_monitoramento-CBM_VG_22-23.xlsx', 'dados_monitoramento-CBM_VG_24-25.xlsx', 'dados_monitoramento-Mae_Bonifacia_22-23.xlsx', 'dados_monitoramento-Mae_Bonifacia_24-25.xlsx', 'dados_monitoramento-UFMT.xlsx']
Sema
BEA CBA
BEA CBA
CBM VG
CBM VG
Mae Bonifacia
Mae Bonifacia
UFMT


In [46]:
def ppb_to_ug(df,pol):

    if pol == 'so2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 2661260.49/10**6        

    elif pol == 'no2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1911038.92/10**6        

    elif pol == 'o3':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1993889.17/10**6        
    
    df.loc[:, "Unidade"] = "ug/m3"

    return df

In [60]:
dict_pols_MT = {
    'co':'CO',
    'no2': 'NO2',
    'so2': 'SO2',
    'o3': 'O3',
    'pm2p5':'MP25',
    'pm10': 'MP10'
}

dict_formatado = {}

for chave in dict_pols_stat.keys():
    
    lista_dfs = dict_pols_stat[chave]
    
    df = pd.concat(lista_dfs, ignore_index=True)

    df["DATETIME"] = pd.to_datetime(
        df.apply(lambda r: f"{r.ANO}-{r.MES}-{r.DIA} {r.HORA}:00:00", axis=1)
    )
    df = df.set_index("DATETIME")

    df = df.sort_index()
    
    lista_horas = pd.date_range(
        start=df.index.min(), 
        end=df.index.max(), 
        freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
    
    if len(lista_horas) != len(df):
        df = df.reindex(pd.DatetimeIndex(lista_horas))

    df['DATETIME'] = df.index

    df = df[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']]
    
    dict_formatado[chave] = df

primeiros_valores = {}

for chave, df in dict_formatado.items():
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_MT = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_MT:
        if data <= codigo_estacao_MT[station]:
            codigo_estacao_MT[station] = data
    else:
        codigo_estacao_MT[station] = data

sorted_items = sorted(
    codigo_estacao_MT.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_MT = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"MT{i:04d}"
    codigo_estacao_MT[nome] = codigo
    
for chave, df in dict_formatado.items():
    
    estacao = codigo_estacao_MT[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == dict_pols_MT[chave.split('_')[-1]], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})

return df_ids

/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = p

SyntaxError: 'return' outside function (2479180702.py, line 85)

In [61]:
df_ids

,ID_OEMA,ID_MMA
0,Mae Bonifacia,MT0001
1,BEA CBA,MT0002
2,CBM VG,MT0003
3,Sema,MT0004
4,UFMT,MT0005


In [185]:
from datetime import timedelta

def fix_24h(row):
    if isinstance(row, str) and row.startswith("24:"):
        # substitui 24: por 00:
        new_str = row.replace("24:", "00:", 1)
        # converte para datetime
        dt = pd.to_datetime(new_str, errors="coerce")
        # adiciona 1 dia
        if pd.notna(dt):
            dt += timedelta(days=1)
        return dt
    else:
        return pd.to_datetime(row, errors="coerce")



In [70]:
dict_stations_MT = {
        'Sema':'CPA - SEMA - CBA',
        'BEA CBA': 'Dom Aquino - BEA - CBA',
        'CBM VG': 'Água Limpa - CBM - VG',
        'Mae Bonifacia': 'Duque de Caxias - Pq Mãe Bonifácia - CBA',
        'UFMT':'Boa Esperança - UFMT - CBA'
    }
    
df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})


df_ids["ID_OEMA"] = df_ids["ID_OEMA"].replace(dict_stations_MT)


df_ids['ID_OEMA']

0    Duque de Caxias - Pq Mãe Bonifácia - CBA
1                      Dom Aquino - BEA - CBA
2                       Água Limpa - CBM - VG
3                            CPA - SEMA - CBA
4                  Boa Esperança - UFMT - CBA
Name: ID_OEMA, dtype: object

In [71]:
df_ids

,ID_OEMA,ID_MMA
0,Duque de Caxias - Pq Mãe Bonifácia - CBA,MT0001
1,Dom Aquino - BEA - CBA,MT0002
2,Água Limpa - CBM - VG,MT0003
3,CPA - SEMA - CBA,MT0004
4,Boa Esperança - UFMT - CBA,MT0005


In [120]:
estado = 'DF'

In [249]:
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

path = path + 'Monitor Report 2024_FINAL.xlsx'

df = pd.read_excel(path)

df.iloc[1] = df.iloc[1].ffill()

poluentes = ['CO_ppm','NO2_ug/m3','NO_ug/m3','NOx_ug/m3','O3_ug/m3','PM10','PM25','PTS','SO2_ug/m3']

dict_pols = {'CO_ppm':'CO',
             'NO2_ug/m3':'NO2',
             'NO_ug/m3':'NO',
             'NOx_ug/m3':'NOX',
             'O3_ug/m3':'O3',
             'PM10':'MP10',
             'PM25':'MP25',
             'PTS':'PTS',
             'SO2_ug/m3':'SO2'}

df.columns = df.iloc[1]

df = df.drop(index=[0, 1]).reset_index(drop=True)

df = df.rename(columns={'Date Time':'DATETIME'})

estacoes = set(df.columns[1:])

dict_pols_stat = defaultdict(list)

for estacao in estacoes:

    df_estacao = df[["DATETIME",estacao]]

    df_estacao.columns = [df_estacao.columns.tolist()[0]] + df_estacao.iloc[0, 1:].tolist()

    df_estacao = df_estacao.drop(index=[0]).reset_index(drop=True)

    for pol in poluentes:

        if pol in df_estacao.columns:
            
            df_pol = df_estacao[["DATETIME",pol]]

            df_pol['UNIDADE'] = df_pol[pol][0]

            df_pol = df_pol.drop(index=[0]).reset_index(drop=True)

            df_pol = df_pol[df_pol["DATETIME"].astype(str).str.contains(r"\d", na=False)].reset_index(drop=True)

            df_pol['DATETIME'] = df_pol['DATETIME'].apply(fix_24h)

            df_pol = df_pol.rename(columns={pol:'VALOR'})

            df_pol.index = df_pol['DATETIME']

            lista_horas = pd.date_range(
                start=df_pol.index.min(), 
                end=df_pol.index.max(), 
                freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
            
            if len(lista_horas) != len(df_pol):
                df_pol = df_pol.reindex(pd.DatetimeIndex(lista_horas))
            
            df_pol['QAQC_INTERNO'] = None
            
            df_pol.insert(1, 'ANO', df_pol.index.year)
            df_pol.insert(2, 'MES', df_pol.index.month)
            df_pol.insert(3, 'DIA', df_pol.index.day)
            df_pol.insert(4, 'HORA', df_pol.index.hour)

            pol = dict_pols[pol]

            df_pol['VALOR'] = pd.to_numeric(df_pol['VALOR'], errors='coerce')

            df_pol = df_pol[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']] 

            dict_pols_stat[estacao+'_'+pol] = df_pol
            

/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pol['UNIDADE'] = df_pol[pol][0]
/tmp/ipykernel_564959/1508039512.py:14: UserWarning: Parsing dates in %d:%M %m/%H/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(row, errors="coerce")
/tmp/ipykernel_564959/901552261.py:57: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

In [258]:
primeiros_valores = {}

for chave, df in dict_pols_stat.items():  
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_DF = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_DF:
        if data <= codigo_estacao_DF[station]:
            codigo_estacao_DF[station] = data
    else:
        codigo_estacao_DF[station] = data

sorted_items = sorted(
    codigo_estacao_DF.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_DF = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"DF{i:04d}"
    codigo_estacao_DF[nome] = codigo
    
for chave, df in dict_pols_stat.items():
    
    estacao = codigo_estacao_DF[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == chave.split('_')[-1], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

dict_oemas = {
    'Estação CRAS FERCAL': 'Fercal CRAS',
    'Estação Escola':	   'Fercal Escola'}

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_DF.keys(),
    'ID_MMA':list(codigo_estacao_DF.values())})

df_ids['ID_OEMA'] = df_ids['ID_OEMA'].replace(dict_oema)

return df_ids

In [275]:
df_ufs = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/IBGE_UFS_CODIGOS.csv')
    
cod_uf =  df_ufs.loc[df_ufs['UF'] == uf, 'CODIGOS'].values[0]

print(cod_uf)

if os.path.exists('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv'):

    df_estacao = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv')

    df_estacao['ID_MMA'] = df_estacao['ID_OEMA'].map(df_ids.set_index('ID_OEMA')['ID_MMA'])

else:

    colunas = ['ID_OEMA', 'UF', 'ID_MMA', 'COD_UF_IBGE', 'CIDADE', 'CD_MUN',
               'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE', 'LATITUDE',
               'LONGITUDE', 'MOBILIDADE', 'REALOCACAO', 'MARCA', 'CATEGORIA',
               'FUNCIONAMENTO', 'METODO', 'FINALIDADE', 'POLUENTE',
               'INICIO', 'STATUS', 'FIM', 'CALIBRACAO', 'OBS_CALIBRACAO', 'MONITORAR',
               'FONTE', 'OBS_GERAIS','DADOS_MONITORAMENTO','RECONHECIDA','REP_ESPACIAL_DECLARADA']
    
    df_estacao = pd.DataFrame(columns=colunas)

df_ids = pol_to_station(df_ids)

mapa = dict(zip(df_ids['ID_MMA'], df_ids['POLUENTE']))

df_estacao['POLUENTE'] = df_estacao['ID_MMA'].map(mapa).fillna(df_estacao['POLUENTE'])

#df_estacao = df_estacao.reindex(df_ids.index)

#df_estacao[["ID_MMA", "ID_OEMA", "POLUENTE"]] = df_ids[["ID_MMA", "ID_OEMA", "POLUENTE"]].values

df_estacao.loc[:, "COD_UF_IBGE"] = cod_uf
df_estacao.loc[:, "UF"] = uf
    
#df_estacao.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes_teste.csv', index=False)

53


In [276]:
df_estacao['POLUENTE']

0                                 PM2,5
1                                  PM10
2                             MP10,MP25
3                                  PM10
4                                  PM10
5                           MP2,5, MP10
6                           MP2,5, MP10
7                           MP2,5, MP10
8    MP10,NO,CO,PTS,O3,SO2,NO2,NOX,MP25
Name: POLUENTE, dtype: object

In [262]:
def pol_to_station(df_ids):

    base_path = Path('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/')

    id_to_poluentes = {}
    
    for poluente_dir in base_path.iterdir():
        if poluente_dir.is_dir():
            poluente = poluente_dir.name
    
            arquivos = [arq.stem for arq in poluente_dir.glob("*")]
    
            for id_mma in df_ids["ID_MMA"]:
                if any(str(arq).startswith(id_mma) for arq in arquivos):
                    id_to_poluentes.setdefault(id_mma, []).append(poluente)
    
    df_ids["POLUENTE"] = df_ids["ID_MMA"].map(id_to_poluentes).fillna("").apply(lambda x: ",".join(x) if isinstance(x, list) else "")
    
    return(df_ids)

# Paraná

In [4]:
df = pd.read_csv(os.getcwd()+'/data/DADOS_BRUTOS/PR/2017/FOZ2017/2017.xls', sep='\t', engine='python', encoding='latin1', header=2 )

df

,Data/Hora,CHUVA(mm),DV(º),PRESS(hPa),RADG(W/m²),TEMP(°C),TEMP INT(°C),UMID(%),VV(m/s),CH4(ppm),...,HCNM(ppm),HCT(ppm),MP10(µg/m³),NO(ppb),NO2(ppb),NOX(ppb),O3(ppb),PTS(µg/m³),SO2(ppb),Unnamed: 20
0,21/06/2017 00:00,insufic,"47,81",insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
1,21/06/2017 01:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
2,21/06/2017 02:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
3,21/06/2017 03:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
4,21/06/2017 04:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4652,31/12/2017 20:00,"0,00","91,25","977,88",insufic,"30,06","30,13","68,28","0,42",insufic,...,insufic,insufic,"54,00","20,56","14,74","35,31","8,72","76,00",insufic,NaN
4653,31/12/2017 21:00,"0,00","50,02","977,78",insufic,"29,45","30,15","73,63","0,47",insufic,...,insufic,insufic,"62,00","0,82","6,98","7,80","10,42","74,00",insufic,NaN
4654,31/12/2017 22:00,"0,00","356,79","978,03",insufic,"28,75","30,16","77,48","0,51",insufic,...,insufic,insufic,"17,00","0,23","6,52","6,75","11,36","13,00",insufic,NaN
4655,31/12/2017 23:00,"0,00","13,49","978,11",insufic,"28,27","30,30","78,45","0,31",insufic,...,insufic,insufic,"41,00","2,15","8,29","10,44","8,64","43,00",insufic,NaN


In [212]:
caminho = os.getcwd()+'/data/DADOS_BRUTOS/PR/2024/'
    
arquivo = 'CIC2024.xls'

if arquivo.endswith(('.xls', '.xlsx')):
    caminho_arquivo = os.path.join(caminho, arquivo)
    try:
        df = pd.read_excel(caminho_arquivo, header=3)
    except Exception as e:
        print(f"Tentando ler como texto: {arquivo}")
        df = pd.read_csv(caminho_arquivo, sep='\t', engine='python', encoding='latin1', header=2)
        print(len(df))
        
    col_data = next(c for c in df.columns if c.startswith('Data'))

    df = (
    df.replace('-', np.nan)
      .assign(**{
          c: pd.to_numeric(
              df[c].astype(str).str.replace(',', '.', regex=False),
              errors='coerce'
          )
          for c in df.columns if c != col_data
      })
      .groupby(col_data, as_index=False)
      .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
    )
    
    if '5MIN' in arquivo:
        estacao = arquivo.split('_')[0]
    else:
        estacao = arquivo.split('2')[0]
    
    if df.columns[0] == 'Data/Hora':

        print(estacao)

Tentando ler como texto: CIC2024.xls
8784


KeyboardInterrupt: 

In [ ]:
print(len(df1))
print(len(df2))

In [213]:
def ler_dados_parana_2024(dicionario,ano):
    
    #caminho =  os.getcwd()+'/data/DADOS_BRUTOS/PR/2024/'

    caminho = os.getcwd()+'/data/DADOS_BRUTOS/PR/'+ano+'/'
    
    arquivos = os.listdir(caminho)
    
    for arquivo in arquivos:
    
        if arquivo.endswith(('.xls', '.xlsx')):
            caminho_arquivo = os.path.join(caminho, arquivo)
            try:
                df = pd.read_excel(caminho_arquivo, header=3)
            except Exception as e:
                print(f"Tentando ler como texto: {arquivo}")
                df = pd.read_csv(caminho_arquivo, sep='\t', engine='python', encoding='latin1', header=2)
                print(len(df))
                
            col_data = next(c for c in df.columns if c.startswith('Data'))
        
            df = (
            df.replace('-', np.nan)
              .assign(**{
                  c: pd.to_numeric(
                      df[c].astype(str).str.replace(',', '.', regex=False),
                      errors='coerce'
                  )
                  for c in df.columns if c != col_data
              })
              .groupby(col_data, as_index=False)
              .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
            )
            
            if '5MIN' in arquivo:
                estacao = arquivo.split('_')[0]
            else:
                estacao = arquivo.split('2')[0]
            
            if df.columns[0] == 'Data/Hora':
        
                print(estacao)
    
                dicionario[ano][estacao] = df

            print(len(df))

    return dicionario

def adicionar_colunas_unidade(df):
    unidades = df.iloc[0]
    
    df = df.iloc[1:].reset_index(drop=True)
    
    for col, unidade in unidades.items():
        if pd.notna(unidade):
            df[f"{col}_UNIDADE"] = unidade
    
    return df

def num_para_hora(valor):
    try:
        h = int(valor)
        m = "30" if valor % 1 == 0.5 else "00"
        return f"{h}:{m}"
    except:
        return None
        
    return df

def ler_dados_parana_1998_2002(dicionario,ano):
    
    caminho = os.getcwd()+'/data/DADOS_BRUTOS/PR/'+ano+'/'
        
    arquivos = os.listdir(caminho)
    
    for arquivo in arquivos:

        print(arquivo)
    
        if any(Path(caminho+arquivo).iterdir()):
            pasta = os.listdir(caminho+arquivo)[0]
    
            df = pd.read_excel(caminho+arquivo+'/'+pasta,header=1)

            df = adicionar_colunas_unidade(df)

            for hora in ['H', 'HORA', 'Hora']:
                if hora in df.columns:
                    df[hora] = df[hora].astype(float).apply(num_para_hora)
                    break

            estacao = arquivo[:-4]
    
            dicionario[ano][estacao] = df 

            print(len(df))
    
        else:
            print('Não há nada em '+ caminho+arquivo)

    return dicionario

def ler_dados_mes_a_mes(caminho):

    tipos_arquivos_ignorar = ['.zip','.rar','.xls','.xlsx','.7z','testes','2016','.ipynb_checkpoints']

    df = pd.DataFrame()

    #print(caminho)
    #print(sorted(os.listdir(caminho)))

    estacao = caminho.split('/')[-2].split('2')[0]
    
    for arquivo in sorted(os.listdir(caminho)):
        df_mes = pd.DataFrame()
    
        if not any(p in arquivo for p in tipos_arquivos_ignorar) or any(p in arquivo for p in ['txt']):

            if '.txt' in arquivo:
                df_mes = pd.read_csv(caminho+arquivo, sep='\t', engine='python', encoding='latin1')
                #print(df_mes.head())
                #print(arquivo)
                df = pd.concat([df, df_mes], ignore_index=True)
            
            else:
                mes = arquivo[:2]
                base_path = os.path.join(caminho, arquivo)
                
                nomes_possiveis = [
                    [f"{estacao}1H_{mes}_{ano}.txt",0],
                    [f"{estacao}1H.txt",0],
                    [f"{estacao}1H_{mes}_{ano}.xls",3],
                    [f"{estacao}_1H.xls",2]
                ]

                for nome in nomes_possiveis:
                    full_path = os.path.join(base_path, nome[0])
                    
                    try:
                        df_mes = pd.read_csv(full_path, sep='\t', engine='python', encoding='latin1',header=nome[1])
                        break
                    except Exception:
                        try:
                            df_mes = pd.read_excel(full_path, engine='xlrd',header=nome[1])
                            break
                        except Exception:
                            continue

            if len(df_mes) == 0:

                print(caminho)
                #print(sorted(os.listdir(caminho)))
                #print(arquivo)
                print(df_mes.head())
                print('')
                
            df = pd.concat([df, df_mes], ignore_index=True)

            #df = pd.concat([df, df_mes], ignore_index=True)
           
    
    print('')
            
    
    return df

def verifica_numero(num):
    try:
        if num != np.nan:
            float(num)
            return True
    except (ValueError, TypeError):
        return False

def verifica_data(data):
    try:
        pd.to_datetime(data)
        return True
    except (ValueError, TypeError):
        return False

def ler_dados_parana_2003_2019(dicionario,ano):

    pastas_ignorar = ['IQA diário','IQA_IAP','2016','ARAUCARIA2018','ARAUCARIA2019','Thumbs.db','~$Validação_Maio_2014.xlsm','SIX1H_2017.zip','.ipynb_checkpoints']

    caminho = os.getcwd()+'/data/DADOS_BRUTOS/PR/'+ano+'/'
        
    arquivos = os.listdir(caminho)

    print('')
    print(ano)
    
    for arquivo in arquivos:

        if arquivo not in pastas_ignorar:

            print(arquivo)
            
            if any(nome.endswith(('.xls', '.xlsx')) for nome in os.listdir(caminho+arquivo)) and len(os.listdir(caminho+arquivo)) <= 3:
                print(os.listdir(caminho+arquivo))

                xlsx = [f for f in os.listdir(caminho+arquivo) if f.endswith('.xlsx')]
                xls = [f for f in os.listdir(caminho+arquivo) if f.endswith('.xls')]
                
                if xlsx:
                    estacao = xlsx[0] 
                elif xls:
                    estacao = xls[0]

                try:
                    if ano == '2003':
                        df = pd.read_excel(caminho+arquivo+'/'+estacao,header=1)
                    elif estacao == 'CIC2019.xlsx':
                        df = pd.read_excel(caminho+arquivo+'/'+estacao,header=2)
                    else:
                        df = pd.read_excel(caminho+arquivo+'/'+estacao)
                except Exception as e:
                    print(f"Tentando ler como texto: {arquivo}")
                    df = pd.read_csv(caminho+arquivo+'/'+estacao, sep='\t', engine='python', encoding='latin1', header=2)
                
                if not (verifica_numero(df[df.columns[0]].iloc[0]) or verifica_data(df[df.columns[0]].iloc[0])) or arquivo == 'SIX2017':

                    df = adicionar_colunas_unidade(df)

                print(len(df))
                    
                dicionario[ano][arquivo[:-4]] = df 

            elif 'IAP' not in arquivo:

                try:
                        
                    df = ler_dados_mes_a_mes(caminho+arquivo+'/')
                    
                    if not (verifica_numero(df[df.columns[0]].iloc[0]) or verifica_data(df[df.columns[0]].iloc[0])):
    
                        df = adicionar_colunas_unidade(df)
                
                    dicionario[ano][arquivo[:-4]] = df 

                    print(len(df))

                except Exception as e:
                    print(f"A seguinte pasta não existe: {caminho}{arquivo}")
                    
    return(dicionario)


In [214]:
len(pd.DataFrame())

0

In [215]:
estacoes_por_ano = {
    '1998': {},
    '1999': {},
    '2000': {},
    '2001': {},
    '2002': {},
    '2003': {},
    '2004': {},
    '2005': {},
    '2006': {},
    '2007': {},
    '2008': {},
    '2009': {},
    '2010': {},
    '2011': {},
    '2012': {},
    '2013': {},
    '2014': {},
    '2015': {},
    '2016': {},
    '2017': {},
    '2018': {},
    '2019': {},
    '2020': {},
    '2021': {},
    '2022': {},
    '2023': {},
    '2024': {}
}

for ano in ['1998','1999','2000',
            '2001','2002','2003','2004','2005','2006','2007','2008','2009','2010',
            '2011','2012','2013','2014','2015','2016','2017','2018','2019','2020',
            '2021','2022','2023','2024']:

    if ano in ['1998','1999','2000','2001','2002']:
        estacoes_por_ano = ler_dados_parana_1998_2002(estacoes_por_ano,ano)
    elif ano in ['2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019']:
        estacoes_por_ano = ler_dados_parana_2003_2019(estacoes_por_ano,ano)
    elif ano in ['2020','2021','2022','2023','2024']:
        estacoes_por_ano = ler_dados_parana_2024(estacoes_por_ano,ano)

STC1998
2969
CIC1998
12067
STC1999
17472
CIC1999
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/1999/CIC1999
CIC2000
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2000/CIC2000
STC2000
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2000/STC2000
ASS2000
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2000/ASS2000
ASS2001
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2001/ASS2001
STC2001
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2001/STC2001
CIC2001
17520
CIC2002
8736
BOQ2002
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2002/BOQ2002
ASS2002
8760

2003
CIC2003
['CIC1h-2003.xls']
8760
IAP2003
CSN2003

8771
UEG2003

6608
PAR2003

8771
RPR2003

3676
BOQ2003

8771

2004
UEG2004

8795
CIC2004

5142
RPR2004

8795
PAR2004

8795
BOQ2004

8795
CSN2004

8795
ASS2004

5142
IAP2004
STC2004

5887

2005
PAR2005

8771

/tmp/ipykernel_273420/3488112494.py:178: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  pd.to_datetime(data)


17519
CIC2017

A seguinte pasta não existe: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2017/CIC2017
BOQ2017
['BOQ1H_2017.xlsx']
8760
PGA2017

8129
PAR2017
['PAR1H_2017.xls']
8760
CSN2017
['CSN1H_2017.xls']
8760
SIX2017
['SIX1H_2017.xls']
5832
ASS2017

A seguinte pasta não existe: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2017/ASS2017
UEG

A seguinte pasta não existe: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2017/UEG

2018
PGA2018

8841
LON2018

17517
CIC2018
['CIC2018.xls']
Tentando ler como texto: CIC2018
8761
ASS2018
['.ipynb_checkpoints', 'ASS18.xls']
Tentando ler como texto: ASS2018
8761
CVEL2018
/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2018/CVEL2018/
Empty DataFrame
Columns: [Report, Date/Time, 1:Temp, 2:O3, 3:CO, 4:NO, 5:NO2, 6:NOx, 7:SO2, 8:CH4 , 9:NMHC, 10:THC, 11:PM10, 12:PTS, 13:AT , 14:RH , 15:BP, 16:SR , 17:WS, 18:WD, 19:RAIN]
Index: []

[0 rows x 21 columns]


14295
CSN2018
['CSN2018.xls']
Tentando ler como t

/tmp/ipykernel_273420/3488112494.py:124: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_mes], ignore_index=True)
/tmp/ipykernel_273420/3488112494.py:158: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_mes], ignore_index=True)


8761
UEG2018
['UEG2018.xls']
Tentando ler como texto: UEG2018
898
FOZ2018

15177
MRGA2018

17519
SIX2018
['.ipynb_checkpoints', 'SIX2018.xls']
Tentando ler como texto: SIX2018
8737

2019
CSN2019
['CSN2019.xls']
Tentando ler como texto: CSN2019
8761
CVEL2019

6783
RPR2019
['RPR2019.xls']
Tentando ler como texto: RPR2019
8761
FOZ2019


/tmp/ipykernel_273420/3488112494.py:124: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_mes], ignore_index=True)
/tmp/ipykernel_273420/3488112494.py:158: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_mes], ignore_index=True)


/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2019/FOZ2019/
Empty DataFrame
Columns: [Report, Date/Time, 1:Temp, 2:O3, 3:CO, 4:NO, 5:NO2, 6:NOx, 7:SO2, 8:CH4 , 9:NMHC, 10:THC, 11:PM10, 12:PTS, 13:AT , 14:RH , 15:BP, 16:SR , 17:WS, 18:WD, 19:RAIN]
Index: []

[0 rows x 21 columns]


9057
LON2019

14457
PGA2019

13341
MRGA2019

17517
ASS2019
['.ipynb_checkpoints', 'ASS2019.xls']
Tentando ler como texto: ASS2019
8761
SIX2019
['SIX2019.xls']
Tentando ler como texto: SIX2019
8737
CIC2019
['CIC2019.xls', 'CIC2019.xlsx']
8761
Tentando ler como texto: RPR2020.xls
8784
RPR
8784
Tentando ler como texto: FIGUACU2020.xls
8784
FIGUACU
8784
Tentando ler como texto: CVEL2020.xls
8784
CVEL
8784
Tentando ler como texto: MRGA2020.xls
8784
MRGA
8784
Tentando ler como texto: SIX2020.xls
8784
SIX
8784
Tentando ler como texto: CSN2020.xls
8784
CSN
8784
Tentando ler como texto: LON2020.xls
8784
LON
8784
Tentando ler como texto: PGA2020.xls
8784
PGA
8784
Tentando ler como texto: CIC2020.xls
8784
CI

In [218]:
lista = []

def parse_datetime(x):
    for fmt in ('%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M', '%m/%d/%Y %I:%M:%S %p'):
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue
    return pd.to_datetime(x, errors='coerce')
                
def criar_datetime(df,tipo):

    if tipo == 'D':

        df['A'] = pd.to_numeric(df['A'], errors='coerce')

        if df['A'].iloc[0] < 2000:

            df['A'] = df['A'] + 2000
                
        df['H'] = df['H'].astype(str).str.split(':').str[0]
        
        df[['A', 'M', 'D', 'H']] = df[['A', 'M', 'D', 'H']].apply(pd.to_numeric, errors='coerce')
        
        df['datetime'] = pd.to_datetime(
            dict(year=df['A'], month=df['M'], day=df['D'], hour=df['H'].clip(upper=23)),
            errors='coerce'
        )
        
        df.loc[df['H'] == 24, 'datetime'] = df.loc[df['H'] == 24, 'datetime'] + pd.Timedelta(hours=1)
    
    elif tipo == 'DATA':

        df['datetime'] = pd.to_datetime(df['DATA']) + pd.to_timedelta(df['HORA'] + ':00')

        df = df.drop(columns=[c for c in ['ANO', 'MES', 'DIA', 'HORA'] if c in df.columns])
    
    elif tipo == 'Data':

        try:
            df['Data'] = pd.to_datetime(df['Data']).dt.date
        except:
            print(1)

        print(df.loc[df['Data'].astype(str).str.contains('--', na=False)])

        df['datetime'] = pd.to_datetime(df['Data']) + pd.to_timedelta(df['Hora'] + ':00')

    elif tipo == 'Data/Hora':
        
        df['Data/Hora'] = df['Data/Hora'].apply(parse_datetime)
        
        df['Data/Hora'] = df['Data/Hora'].dt.strftime('%Y-%m-%d %H:%M:%S')

        df['datetime'] = pd.to_datetime(df['Data/Hora'])

    elif tipo == 'Date/Time':
        
        df['Date/Time'] = df['Date/Time'].apply(parse_datetime)
        
        df['Date/Time'] = df['Date/Time'].dt.strftime('%Y-%m-%d %H:%M:%S')

        df['datetime'] = pd.to_datetime(df['Date/Time'])
        
    df = df.set_index("datetime")

    df = df.sort_index()
    
    df.insert(0, 'DATETIME', df.index)
    df.insert(1, 'ANO', df.index.year)
    df.insert(2, 'MES', df.index.month)
    df.insert(3, 'DIA', df.index.day)
    df.insert(4, 'HORA', df.index.hour)

    return df

lista_colunas = []

for ano in estacoes_por_ano.keys():
    
    print(ano)

    for estacao in estacoes_por_ano[ano].keys():

        #print(estacao)
        
        lista.append(estacao)

        lista_colunas = lista_colunas + list(estacoes_por_ano[ano][estacao].columns)
        
        if any(item in ['D','DATA','Data','Date/Time','Data/Hora'] for item in estacoes_por_ano[ano][estacao].columns):

            if 'D' in estacoes_por_ano[ano][estacao].columns:
                print('D')
                estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'D')
                #print(estacoes_por_ano[ano][estacao])
            elif 'DATA' in estacoes_por_ano[ano][estacao].columns:
                print('DATA')
                estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'DATA')
                #print(estacoes_por_ano[ano][estacao])
            elif 'Data' in estacoes_por_ano[ano][estacao].columns:
                print('Data')
                estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'Data')
                #print(estacoes_por_ano[ano][estacao])
            elif 'Date/Time' in estacoes_por_ano[ano][estacao].columns:
                print('Date/Time')
                estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'Date/Time')
                #print(estacoes_por_ano[ano][estacao])
            elif 'Data/Hora' in estacoes_por_ano[ano][estacao].columns:
                print('Data/Hora')
                estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'Data/Hora')
                #print(estacoes_por_ano[ano][estacao])
            else:
                print('ERRO')
    print('')

'''
lista1 = sorted(set(lista))

lista_pr = pd.read_csv(os.getcwd() + '/data/DADOS_ESTACOES/PR_estacoes.csv')

lista2 = sorted(lista_pr['ID_OEMA'])

iguais = sorted(set(lista1) & set(lista2))

so_lista1 = [x for x in lista1 if x not in iguais]
so_lista2 = [x for x in lista2 if x not in iguais]

col1 = iguais + so_lista1 + [np.nan] * len(so_lista2)
col2 = iguais + [np.nan] * len(so_lista1) + so_lista2

df = pd.DataFrame({'PR_dados': col1, 'PR_estacao': col2})

df'''

#print(lista_colunas)
print(len(list(set(lista_colunas))))

1998
Data
Empty DataFrame
Columns: [Data, Hora, SO2, NO, NO2, Nox, O3, UVB, Temperatura, Umidade, Rad. Glob.,  UVA, Press, V V, D V, SO2_UNIDADE, NO_UNIDADE, NO2_UNIDADE, Nox_UNIDADE, O3_UNIDADE, UVB_UNIDADE, Temperatura_UNIDADE, Umidade_UNIDADE, Rad. Glob._UNIDADE,  UVA_UNIDADE, Press_UNIDADE, V V_UNIDADE, D V_UNIDADE]
Index: []

[0 rows x 28 columns]
Data
Empty DataFrame
Columns: [Data, Hora, SO2, NO, NO2, Nox, O3, Temperatura, Umidade, Rad. Glob., Rad. UV, Press, V V, D V, SO2_UNIDADE, NO_UNIDADE, NO2_UNIDADE, Nox_UNIDADE, O3_UNIDADE, Temperatura_UNIDADE, Umidade_UNIDADE, Rad. Glob._UNIDADE, Rad. UV_UNIDADE, Press_UNIDADE, V V_UNIDADE, D V_UNIDADE]
Index: []

[0 rows x 26 columns]

1999
Data
Empty DataFrame
Columns: [Data, Hora, SO2, NO, NO2, Nox, O3, UVB, Temperatura, Umidade, Rad. Glob.,  UVA, Press, V V, D V, SO2_UNIDADE, NO_UNIDADE, NO2_UNIDADE, Nox_UNIDADE, O3_UNIDADE, UVB_UNIDADE, Temperatura_UNIDADE, Umidade_UNIDADE, Rad. Glob._UNIDADE,  UVA_UNIDADE, Press_UNIDADE, V V_UNIDAD

In [329]:
pol_estacao_ano['2015']['CIC']['NO']

,DATETIME,ANO,MES,DIA,HORA,VALOR,UNIDADE,QAQC_INTERNO
datetime,,,,,,,,
2015-01-01 01:00:00,2015-01-01 01:00:00,2015.0,1.0,1.0,1.0,"1,29",ppb,NaN
2015-01-01 02:00:00,2015-01-01 02:00:00,2015.0,1.0,1.0,2.0,"0,95",ppb,NaN
2015-01-01 03:00:00,2015-01-01 03:00:00,2015.0,1.0,1.0,3.0,"0,63",ppb,NaN
2015-01-01 04:00:00,2015-01-01 04:00:00,2015.0,1.0,1.0,4.0,"0,78",ppb,NaN
2015-01-01 05:00:00,2015-01-01 05:00:00,2015.0,1.0,1.0,5.0,"1,34",ppb,NaN
...,...,...,...,...,...,...,...,...
NaT,NaT,NaN,NaN,NaN,NaN,ppb,ppb,NaN
NaT,NaT,NaN,NaN,NaN,NaN,ppb,ppb,NaN
NaT,NaT,NaN,NaN,NaN,NaN,ppb,ppb,NaN


    ANO  MES  DIA  HORA UNIDADE  VALOR
0  2023    5   10    12     ppb   41.0
1  2023    5   10    13     ppb   44.0


,ANO,MES,DIA,HORA,UNIDADE,VALOR,DATETIME
DATETIME,,,,,,,
NaT,1998.0,11.0,1.0,0.0,ppb,1.555,1998-11-01 00:00:00
NaT,1998.0,11.0,1.0,1.0,ppb,1.515,1998-11-01 01:00:00
NaT,1998.0,11.0,1.0,2.0,ppb,1.630,1998-11-01 02:00:00
NaT,1998.0,11.0,1.0,3.0,ppb,1.660,1998-11-01 03:00:00
NaT,1998.0,11.0,1.0,4.0,ppb,1.525,1998-11-01 04:00:00
...,...,...,...,...,...,...,...
NaT,2016.0,12.0,31.0,20.0,ppb,-99.000,2016-12-31 20:00:00
NaT,2016.0,12.0,31.0,21.0,ppb,-99.000,2016-12-31 21:00:00
NaT,2016.0,12.0,31.0,22.0,ppb,-99.000,2016-12-31 22:00:00


In [219]:
df_col_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/colunas_pol_PR.csv',encoding='UTF-8')

pol_estacao_ano = {}

print(df_col_pols)

for ano in estacoes_por_ano.keys():

    pol_estacao_ano[ano] = {}

    for estacao in estacoes_por_ano[ano]:
    
        pol_estacao_ano[ano][estacao] = {}
        
        for pol in df_col_pols['col_pr']:

            if pol in estacoes_por_ano[ano][estacao].columns and 'UNIDADE' not in pol:

                poluente_mma = df_col_pols.loc[df_col_pols['col_pr'] == pol, 'col_mma'].iloc[0]

                qaqc_interno = np.nan
                
                if '(' in pol:
                    unidade = pol.split('(')[-1][:-1]
                    print(pol)            
                    #print(unidade)

                    df = estacoes_por_ano[ano][estacao][['DATETIME','ANO','MES','DIA','HORA',pol]]

                    df['UNIDADE'] = unidade
                    df['QAQC_INTERNO'] = qaqc_interno

                    df.rename(columns={pol: 'VALOR'}, inplace=True)
                    
                elif (pol+'_UNIDADE') in estacoes_por_ano[ano][estacao].columns:
                    #print('')
                    #print(pol)
                    #print('')

                    df = estacoes_por_ano[ano][estacao][['DATETIME','ANO','MES','DIA','HORA',pol,pol+'_UNIDADE']]

                    df['QAQC_INTERNO'] = qaqc_interno

                    df.rename(columns={pol: 'VALOR',pol+'_UNIDADE':'UNIDADE'}, inplace=True)
                    
                else:

                    df = estacoes_por_ano[ano][estacao][['DATETIME','ANO','MES','DIA','HORA',pol]]
                    
                    df['UNIDADE'] = np.nan
                    df['QAQC_INTERNO'] = qaqc_interno

                    df.rename(columns={pol: 'VALOR'}, inplace=True)

                pol_estacao_ano[ano][estacao][poluente_mma] = df

        

            col_pr          col_mma
0           9:NMHC             HCNM
1     2:O3_UNIDADE       O3_UNIDADE
2      TOL_UNIDADE  TOLUENO_UNIDADE
3               PI             MP10
4         SO2(ppb)              SO2
..             ...              ...
71     Nox_UNIDADE      NOX_UNIDADE
72             BEN          BENZENO
73            2:O3               O3
74        NOX(ppb)              NOX
75  12:PTS_UNIDADE              PTS

[76 rows x 2 columns]


/tmp/ipykernel_273420/317706924.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['QAQC_INTERNO'] = qaqc_interno
/tmp/ipykernel_273420/317706924.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={pol: 'VALOR',pol+'_UNIDADE':'UNIDADE'}, inplace=True)
/tmp/ipykernel_273420/317706924.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

SO2(ppb)
MP10(µg/m³)
H2S(ppb)
PTS(µg/m³)
TRS(ppb)
CS(ppb)


/tmp/ipykernel_273420/317706924.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['QAQC_INTERNO'] = qaqc_interno
/tmp/ipykernel_273420/317706924.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={pol: 'VALOR',pol+'_UNIDADE':'UNIDADE'}, inplace=True)
/tmp/ipykernel_273420/317706924.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

SO2(ppb)
CH4(ppm)
O3(ppb)
NO(ppb)
HCNM(ppm)
MP10(µg/m³)
NO2(ppb)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
NOX(ppb)


...

/tmp/ipykernel_273420/317706924.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['UNIDADE'] = unidade
/tmp/ipykernel_273420/317706924.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['QAQC_INTERNO'] = qaqc_interno
/tmp/ipykernel_273420/317706924.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(co

MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
SO2(ppb)
O3(ppb)
NO(ppb)
NO2(ppb)
PTS(µg/m³)
NOX(ppb)
SO2(ppb)
CH4(ppm)
O3(ppb)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
CH4(ppm)
O3(ppb)
NO(ppb)
HCNM(ppm)
MP10(µg/m³)
NO2(ppb)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
NOX(ppb)
SO2(ppb)
O3(ppb)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
NOX(ppb)
SO2(ppb)
CH4(ppm)
O3(ppb)
NO(ppb)
HCNM(ppm)
MP10(µg/m³)
NO2(ppb)
CO(ppm)
HCT(ppm)
NOX(ppb)
CH4(ppm)
NO(ppb)
HCNM(ppm)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
HCT(ppm)
NOX(ppb)
SO2(ppb)
O3(ppb)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
HCT(ppm)
NOX(ppb)
SO2(ppb)
CH4(ppm)
NO(ppb)
HCNM(ppm)
NO2(ppb)
CO(ppm)
HCT(ppm)
NOX(ppb)
SO2(ppb)
CH4(ppm)
O3(ppb)
NO(ppb)


/tmp/ipykernel_273420/317706924.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['QAQC_INTERNO'] = qaqc_interno
/tmp/ipykernel_273420/317706924.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={pol: 'VALOR'}, inplace=True)
/tmp/ipykernel_273420/317706924.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-vers

MP10(µg/m³)
NO2(ppb)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
SO2(ppb)
CH4(ppm)
O3(ppb)
NO(ppb)
HCNM(ppm)
MP10(µg/m³)
NO2(ppb)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
NOX(ppb)
SO2(ppb)
O3(ppb)
NO(ppb)
NO2(ppb)
PTS(µg/m³)
NOX(ppb)
SO2(ppb)
CH4(ppm)
O3(ppb)
NO(ppb)
HCNM(ppm)
MP10(µg/m³)
NO2(ppb)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
NOX(ppb)
SO2(ppb)
O3(ppb)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
NOX(ppb)
SO2(ppb)
O3(ppb)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
SO2(ppb)
MP10(µg/m³)
PM_2_5(µg/m³)
CO(ppm)
TRS(ppb)
PM_2_5(µg/m³)
PM_2_5(µg/m³)
PM_2_5(µg/m³)
PTS(µg/m³)
SO2(ppb)
O3(ppb)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
NOX(ppb)
PM_2_5(µg/m³)
HCNM(ppm)
MP10(µg/m³)
PM_2_5(µg/m³)
PTS(µg/m³)
HCT(ppm)
CS(ppb)
PM_2_5(µg/m³)
SO2(ppb)
O3(ppb)
NO(ppb)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
CO(ppm)
SO2(ppb)
CH4(ppm)
O3(ppb)
NO(ppb)
HCNM(ppm)
MP10(µg/m³)
NO2(ppb)
PM_2_5(µg/m³)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
NOX(ppb)
PM_2_5(µg/m³)
SO2(ppb)
CH4(ppm)
NO(ppb)
HCNM(ppm)
NO2(ppb)
PTS(µg/m³)
CO(ppm)
HCT(ppm)
NOX

/tmp/ipykernel_273420/317706924.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['UNIDADE'] = unidade
/tmp/ipykernel_273420/317706924.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['QAQC_INTERNO'] = qaqc_interno
/tmp/ipykernel_273420/317706924.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(co

In [231]:
import pandas as pd

pol_estacao = {}

for ano, estacoes in pol_estacao_ano.items():
    for estacao, poluentes in estacoes.items():
        for poluente, df in poluentes.items():
            pol_estacao.setdefault(estacao, {}).setdefault(poluente, [])
            pol_estacao[estacao][poluente].append(df)

estacoes_finais = {}

for estacao, poluentes in pol_estacao.items():
    for poluente, lista_dfs in poluentes.items():
        df_concat = pd.concat(lista_dfs, ignore_index=True)

        df_concat = df_concat[df_concat['DATETIME'].notna()]

        df_concat['VALOR'] = (
            df_concat['VALOR']
            .astype(str)           
            .str.replace(',', '.', regex=False)  
        )

        df_concat['VALOR'] = pd.to_numeric(df_concat['VALOR'], errors='coerce')

        df_media = (
            df_concat.groupby(['ANO', 'MES', 'DIA', 'HORA', 'UNIDADE'], as_index=False)['VALOR']
              .mean()
        )
        
        df_media["DATETIME"] = pd.to_datetime(
            df_media.apply(
                lambda r: f"{int(r.ANO):04d}-{int(r.MES):02d}-{int(r.DIA):02d} {int(r.HORA):02d}:00:00",
                axis=1
            )
        )

        df_media = df_media.drop(columns=['ANO', 'MES', 'DIA', 'HORA'])

        df_media = df_media.set_index("DATETIME")
    
        df_media = df_media.sort_index()

        df_media = df_media[~df_media.index.duplicated(keep='first')]
        
        lista_horas = pd.date_range(
            start=df_media.index.min(), 
            end=df_media.index.max(), 
            freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
        
        if len(lista_horas) != len(df):
            df_media = df_media.reindex(pd.DatetimeIndex(lista_horas))

        df_media.insert(1, 'ANO', df_media.index.year)
        df_media.insert(2, 'MES', df_media.index.month)
        df_media.insert(3, 'DIA', df_media.index.day)
        df_media.insert(4, 'HORA', df_media.index.hour)
        
        df_media['DATETIME'] = df_media.index

        df_media['QAQC_INTERNO'] = None
        
        df_media = df_media[['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR', 'UNIDADE', 'QAQC_INTERNO']]

        estacoes_finais[estacao+'_'+poluente] = df_media

/tmp/ipykernel_273420/3105762284.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_273420/3105762284.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_273420/3105762284.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_273420/3105762284.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_273420/3105762284.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_273420/3105762284.py:47: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = p

In [232]:
estacoes_finais.keys()

dict_keys(['STC_NO', 'STC_O3', 'STC_NO2', 'STC_NOX', 'STC_SO2', 'STC_MP10', 'STC_CH4', 'STC_TOLUENO', 'STC_PTS', 'STC_HCT', 'STC_ETILBENZENO', 'STC_NH3', 'STC_CO', 'STC_HCNM', 'STC_BENZENO', 'CIC_NO', 'CIC_O3', 'CIC_NO2', 'CIC_NOX', 'CIC_SO2', 'CIC_PTS', 'CIC_CH4', 'CIC_HCT', 'CIC_HCNM', 'CIC_MP10', 'CIC_TOLUENO', 'CIC_ETILBENZENO', 'CIC_NH3', 'CIC_CO', 'CIC_BENZENO', 'CIC_MP25', 'ASS_NO', 'ASS_PTS', 'ASS_O3', 'ASS_CH4', 'ASS_HCT', 'ASS_NO2', 'ASS_NOX', 'ASS_SO2', 'ASS_HCNM', 'ASS_MP10', 'ASS_TOLUENO', 'ASS_ETILBENZENO', 'ASS_NH3', 'ASS_CO', 'ASS_BENZENO', 'CSN_MP10', 'CSN_NO', 'CSN_O3', 'CSN_CH4', 'CSN_TOLUENO', 'CSN_PTS', 'CSN_HCT', 'CSN_NO2', 'CSN_ETILBENZENO', 'CSN_SO2', 'CSN_NH3', 'CSN_CO', 'CSN_HCNM', 'CSN_BENZENO', 'CSN_NOX', 'UEG_MP10', 'UEG_NO', 'UEG_O3', 'UEG_CH4', 'UEG_TOLUENO', 'UEG_PTS', 'UEG_HCT', 'UEG_NO2', 'UEG_ETILBENZENO', 'UEG_SO2', 'UEG_NH3', 'UEG_CO', 'UEG_HCNM', 'UEG_BENZENO', 'PAR_MP10', 'PAR_NO', 'PAR_O3', 'PAR_CH4', 'PAR_TOLUENO', 'PAR_PTS', 'PAR_HCT', 'PAR_NO2

In [233]:

def create_QAQCMMA_VALOR(df,pol):

    df = df.rename(columns={'VALOR':'VALOR_ORIGINAL'})

    flags_invalidos = ['!', 'IF', 'IO', 'IC', 'I%', 'IL', 'IE', 'IS', 'IU', 'IM', 'IP', 'ID', 'IT', 'IR', 
                       'Fora da Faixa de Medição', 'Disabilitada Temporariamente', 'Inválido', 
                       'Insuficientes', 'Inexistente']

    df['QAQC_INTERNO'] = ~df['QAQC_INTERNO'].isin(flags_invalidos)
    
    DEFAULT_RANGE_LIMITS = {
        "O3": (0, 500),
        "CO": (0, 50),
        "NO2": (0, 1000),
        "NOX": (0, 2000),
        "SO2": (0, 1000),
        "MP25": (0, 1000),
        "MP10": (0, 2000),
    }

    df['QAQC_MMA'] = df['QAQC_INTERNO']

    if pol in list(DEFAULT_RANGE_LIMITS.keys()):
        lim_min = DEFAULT_RANGE_LIMITS[pol][0]
        lim_max = DEFAULT_RANGE_LIMITS[pol][1]
    else:
        lim_min = 0
        lim_max = np.inf

    df['VALOR'] = df['VALOR_ORIGINAL']

    df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce')
    
    df.loc[df['QAQC_MMA'] & (df['VALOR'].isna() | (df['VALOR'] <= lim_min) | (df['VALOR'] >= lim_max)), 'QAQC_MMA'] = False
    
    df.loc[~df['QAQC_MMA'], 'VALOR'] = np.nan
    
    df = df[['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR', 'VALOR_ORIGINAL', 'UNIDADE', 'QAQC_INTERNO', 'QAQC_MMA']]

    return df


primeiros_valores = {}

for chave, df in estacoes_finais.items():  
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_PR = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_PR:
        if data <= codigo_estacao_PR[station]:
            codigo_estacao_PR[station] = data
    else:
        codigo_estacao_PR[station] = data

sorted_items = sorted(
    codigo_estacao_PR.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_PR = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"PR{i:04d}"
    codigo_estacao_PR[nome] = codigo

for chave, df in estacoes_finais.items():
    
    estacao = codigo_estacao_PR[chave.split('_')[0]]

    if chave.split('_')[-1] != 'CS':
        
        cod_pol = tabela_pols.loc[tabela_pols['NOME_PASTA'] == chave.split('_')[-1], 'COD_POLUENTE'].values[0]
        
        nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
        df = create_QAQCMMA_VALOR(df,nome_pasta)

        df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(int(cod_pol)).zfill(3)+'.csv',index=False)

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_PR.keys(),
    'ID_MMA':list(codigo_estacao_PR.values())})

print(df_ids)

    ID_OEMA  ID_MMA
0       CIC  PR0001
1       STC  PR0002
2       ASS  PR0003
3       BOQ  PR0004
4       CSN  PR0005
5       PAR  PR0006
6       UEG  PR0007
7       RPR  PR0008
8       SIX  PR0009
9       CAS  PR0010
10      PGA  PR0011
11      LON  PR0012
12     MRGA  PR0013
13      FOZ  PR0014
14     CVEL  PR0015
15  FIGUACU  PR0016
16     PARP  PR0017
17   FOSPAR  PR0018
18      GPC  PR0019
19      MGA  PR0020
20  FNB.xls  PR0021
21      GRP  PR0022
22      LDA  PR0023
23      CLB  PR0024
24      JDA  PR0025


In [234]:
df_ids

,ID_OEMA,ID_MMA
0,CIC,PR0001
1,STC,PR0002
2,ASS,PR0003
3,BOQ,PR0004
4,CSN,PR0005
5,PAR,PR0006
6,UEG,PR0007
7,RPR,PR0008
8,SIX,PR0009
9,CAS,PR0010


In [413]:
df_ids

,ID_OEMA,ID_MMA
0,CIC,PR0001
1,STC,PR0002
2,ASS,PR0003
3,BOQ,PR0004
4,CSN,PR0005
5,PAR,PR0006
6,UEG,PR0007
7,RPR,PR0008
8,SIX,PR0009
9,CAS,PR0010


In [225]:
monitoramento_QAr_BR.loc[monitoramento_QAr_BR['ID_OEMA'] == nome, 'ID_MMA']

Series([], Name: ID_MMA, dtype: object)

In [30]:
estacoes_dicionario.keys()

for chave in estacoes_dicionario.keys():

    print(chave)
    print(len(estacoes_dicionario[chave]))
    print(estacoes_dicionario[chave].columns[0])
    print(estacoes_dicionario[chave].iloc[0, 0])
    print(estacoes_dicionario[chave].iloc[-1, 0])
    print("")

    estacoes_por_ano

,D,M,A,H,SO2,NO,NO2,O3,CO,PTS,...,THC,CH4,NMHC,TEMP,UMID,PRESS,VV,DV,RADG,CHUVA
0,NaN,NaN,NaN,NaN,ppb,ppb,ppb,ppb,ppm,µg/m³,...,ppm,ppm,ppm,°C,%,mBar,m/s,graus,W/m2,mm
1,1.0,1.0,17.0,1.0,0,NaN,NaN,7.8,0.46,NaN,...,1.39,1.33,NaN,23.2,99.9,923.3,1.2,232.4,7.7,NaN
2,1.0,1.0,17.0,2.0,0.2,NaN,NaN,15,0.38,NaN,...,1.38,1.26,NaN,22,99.9,922.2,1.2,189.2,6,NaN
3,1.0,1.0,17.0,3.0,0.6,NaN,NaN,10.7,0.37,NaN,...,1.35,1.26,NaN,21.9,99.9,921.5,0.9,274.3,5.3,NaN
4,1.0,1.0,17.0,4.0,0.4,NaN,NaN,11.1,0.41,NaN,...,1.35,1.26,NaN,21.7,99.9,921.1,1.2,275,4,NaN


# Update_MQAr_with_IEMA_MonitorAr

In [427]:
caminho = os.getcwd() + '/data/DADOS_BRUTOS/BRUTO_IEMA/'

estados_iema = os.listdir(caminho)

df_estacao_codigo = pd.DataFrame({
    'Estacao':[],
    'Codigo':[]
})

for estado in estados_iema:

    if '.csv' not in estado and '.ipynb_checkpoints' not in estado:

        lista_anos = os.listdir(caminho+estado)

        print(estado)
    
        print(lista_anos)
    
        df_estado = pd.DataFrame({
            'Data':[], 
            'Hora':[], 
            'Estacao':[], 
            'Codigo':[], 
            'Poluente':[], 
            'Valor':[], 
            'Unidade':[],
            'Tipo':[]})
    
        for ano in lista_anos:

            if '.csv' in ano:

                df = pd.read_csv(caminho+estado+'/'+ano, encoding='ISO-8859-1')
        
                df.columns.values[0] = 'Data'

                if 'Escola CecÃlia Meireles' in df['Estacao']:

                    print(ano)
        
                df_estado = pd.concat([df_estado, df], ignore_index=True)
    
        df_estacao_codigo = pd.concat([df_estacao_codigo,df_estado[['Estacao','Codigo']]], ignore_index=True)

df_estacao_codigo = df_estacao_codigo.drop_duplicates(subset=['Estacao', 'Codigo']).reset_index(drop=True)


DF
['DF2017.csv', 'DF2016.csv', 'DF2020.csv', 'DF2015.csv', 'DF2018.csv', 'DF2022.csv', 'DF2021.csv', 'DF2019.csv']
GO
['GO2015.csv', 'GO2017.csv', 'GO2016.csv']
PR
['PR2016.csv', 'PR2015.csv', 'PR2020.csv', 'PR2018.csv', 'PR2017.csv', 'PR2019.csv']
MG
['MG2016.csv', 'MG2022.csv', 'MG2021.csv', 'MG2018.csv', 'MG2017.csv', 'MG2019.csv', 'MG2015.csv', 'MG2020.csv']
RJ
['RJ202001.csv', 'RJ202201.csv', 'RJ201602.csv', 'RJ201902.csv', 'RJ201804.csv', 'RJ201502.csv', 'RJ201503.csv', 'RJ202101.csv', 'RJ201701.csv', 'RJ202102.csv', 'RJ201802.csv', 'RJ202002.csv', 'RJ201803.csv', 'RJ202202.csv', 'RJ201801.csv', 'RJ201702.csv', 'RJ201501.csv', 'RJ202103.csv', 'RJ201601.csv', 'RJ201901.csv']
SP
['SP201501.csv', 'SP201701.csv', 'SP201502.csv', 'SP202202.csv', 'SP202101.csv', 'SP201802.csv', 'SP201601.csv', 'SP201901.csv', 'SP202001.csv', 'SP201602.csv', 'SP202102.csv', 'SP202002.csv', 'SP201801.csv', 'SP202201.csv', 'SP201902.csv', 'SP201702.csv']
PE
['PE2020.csv', 'PE2021.csv', 'PE2017.csv', 'PE2

In [428]:
df_estacao_codigo.to_csv(os.getcwd() + '/data/DADOS_BRUTOS/BRUTO_IEMA/IEMA_codigo_estacao.csv', encoding = 'UTF-8', index = False)

In [439]:
df_estacoes = pd.read_csv(os.getcwd() + '/data/DADOS_BRUTOS/BRUTO_IEMA/IEMA_codigo_estacao.csv', encoding = 'UTF-8')

import pandas as pd
import glob
import os
from unidecode import unidecode

# Caminho da pasta com os arquivos CSV
pasta = os.getcwd() + '/data/DADOS_ESTACOES'

# DataFrame principal
df_principal = df_estacoes.copy()

# Garantir que são strings
df_principal["ID_OEMA"] = df_principal["ID_OEMA"].astype(str)
df_principal["ID_MMA"] = df_principal["ID_MMA"].astype(str)

# Função para remover acentos e padronizar texto
def normalizar(texto):
    if pd.isna(texto):
        return None
    return unidecode(str(texto)).strip().upper()

# Normalizar coluna no DataFrame principal
df_principal["ID_OEMA_norm"] = df_principal["ID_OEMA"].apply(normalizar)

# Criar dicionário de mapeamento OEMA → MMA
mapa_oema_mma = {}

for arquivo in glob.glob(os.path.join(pasta, "*.csv")):
    df = pd.read_csv(arquivo, dtype=str)
    
    if "ID_OEMA" in df.columns and "ID_MMA" in df.columns:
        # Normalizar também nos arquivos
        df["ID_OEMA_norm"] = df["ID_OEMA"].apply(normalizar)
        df["ID_MMA"] = df["ID_MMA"].astype(str)

        pares_validos = df.dropna(subset=["ID_OEMA_norm", "ID_MMA"])
        for _, linha in pares_validos.iterrows():
            mapa_oema_mma[linha["ID_OEMA_norm"]] = linha["ID_MMA"]

# Mapear no DataFrame principal (usando os IDs normalizados)
df_principal["ID_MMA_preenchido"] = df_principal["ID_OEMA_norm"].map(mapa_oema_mma)

# Preencher apenas onde estava vazio
df_principal["ID_MMA"] = df_principal["ID_MMA"].replace("nan", pd.NA)
df_principal["ID_MMA"] = df_principal["ID_MMA"].fillna(df_principal["ID_MMA_preenchido"])

# Remover coluna auxiliar
df_principal = df_principal.drop(columns=["ID_MMA_preenchido", "ID_OEMA_norm"])

df_principal.to_csv(os.getcwd() + '/data/DADOS_BRUTOS/BRUTO_IEMA/IEMA_codigo_estacao_teste.csv', encoding = 'UTF-8', index = False)


In [440]:
df_estacoes = pd.read_csv(os.getcwd() + '/data/DADOS_BRUTOS/BRUTO_IEMA/IEMA_codigo_estacao_teste.csv', encoding = 'UTF-8')

df_estacoes

,ID_OEMA,ID_IEMA,ID_MMA
0,Rodoviária do Plano Piloto,DF03,NaN
1,Fercal I,DF01,NaN
2,CIPLAN (antiga estação Queima Lençol),DF02,NaN
3,Setor Comercial Sul,DF05,NaN
4,Jardim Zoológico de Brasília,DF04,NaN
...,...,...,...
416,Triunfo - DEPREC,RS04,RS0009
417,Charqueadas - Arranca Toco,RS06,RS0013
418,Rio Grande Porto - FURG - Movel,RS18,RS0018
419,Enseada do Suá,ES03,ES0003


In [457]:
caminho = os.getcwd() + '/data/DADOS_BRUTOS/BRUTO_IEMA/'

df_est = pd.DataFrame({'Data':[],
                   'Hora'    :[],	
                   'Estacao' :[],
                   'Codigo'  :[],
                   'Poluente':[],
                   'Valor'   :[],	
                   'Unidade' :[],
                   'Tipo'    :[]})

arquivos = os.listdir(caminho)

for arquivo in arquivos:
    
    if '.csv' not in arquivo and 'checkpoints' not in arquivo:

        print(arquivo)

        anos = os.listdir(caminho+arquivo)

        for ano in anos:

            if '.csv' in ano:

                df = pd.read_csv(caminho+arquivo+'/'+ano, encoding='ISO-8859-1')

                df = df.rename(columns=lambda x: x.replace('ï»¿', ''))

                df = df.replace('MP2.5', 'MP25', regex=False)

                df_est = pd.concat([df, df_est], ignore_index=True)


DF
GO
PR
MG
RJ
SP
PE
BA
CE
ES
RS


In [458]:
df_est = df_est.merge(df_estacoes[['ID_OEMA', 'ID_MMA']], left_on='Estacao', right_on='ID_OEMA', how='left')

In [515]:
list(set(df_est['Unidade']))

['ug/m3', 'ppb', 'µg/m3', 'Âµg/mÂ³', 'ppm', 'µg/m³']

In [475]:
tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')

In [204]:
lista_pols = list(set(df_est['Poluente']))


def create_QAQCMMA_VALOR(df,pol):

    df = df.rename(columns={'VALOR':'VALOR_ORIGINAL'})

    flags_invalidos = ['!', 'IF', 'IO', 'IC', 'I%', 'IL', 'IE', 'IS', 'IU', 'IM', 'IP', 'ID', 'IT', 'IR',
                       'IM', 'VU', 'ID', 'IF', 'VE', 'IC', 'IN', 'VR', 'IV', 'IO',
                       'Fora da Faixa de Medição', 'Disabilitada Temporariamente', 'Inválido', 
                       'Insuficientes', 'Inexistente']

    df['QAQC_INTERNO'] = ~df['QAQC_INTERNO'].isin(flags_invalidos)
    
    DEFAULT_RANGE_LIMITS = {
        "O3": (0, 500),
        "CO": (0, 50),
        "NO2": (0, 1000),
        "NOX": (0, 2000),
        "SO2": (0, 1000),
        "MP25": (0, 1000),
        "MP10": (0, 2000),
    }

    df['QAQC_MMA'] = df['QAQC_INTERNO']

    if pol in list(DEFAULT_RANGE_LIMITS.keys()):
        lim_min = DEFAULT_RANGE_LIMITS[pol][0]
        lim_max = DEFAULT_RANGE_LIMITS[pol][1]
    else:
        lim_min = 0
        lim_max = np.inf

    df['VALOR'] = df['VALOR_ORIGINAL']

    df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce')
    
    df.loc[df['QAQC_MMA'] & (df['VALOR'].isna() | (df['VALOR'] <= lim_min) | (df['VALOR'] >= lim_max)), 'QAQC_MMA'] = False
    
    df.loc[~df['QAQC_MMA'], 'VALOR'] = np.nan
    
    df = df[['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR', 'VALOR_ORIGINAL', 'UNIDADE', 'QAQC_INTERNO', 'QAQC_MMA']]

    return df

print(lista_pols)

def ajustar_hora24(row):
    if row['Hora'].startswith('24'):
        # passa para o dia seguinte
        nova_data = pd.to_datetime(row['Data']) + pd.Timedelta(days=1)
        return nova_data.strftime('%Y-%m-%d') + ' 00:00:00'
    else:
        return row['Data'] + ' ' + row['Hora']

def ug_to_ppm(df):

    df.loc[df["UNIDADE"] == "ug/m3", "VALOR_ORIGINAL"] *= 868.26/10**6
    df.loc[df["UNIDADE"] == "Âµg/mÂ", "VALOR_ORIGINAL"] *= 868.26/10**6
    df.loc[df["UNIDADE"] == "µg/m³", "VALOR_ORIGINAL"] *= 868.26/10**6
    df.loc[df["UNIDADE"] == "µg/m3", "VALOR_ORIGINAL"] *= 868.26/10**6
    df.loc[df["UNIDADE"] == "ppb", "VALOR_ORIGINAL"] *= 1/1000

    df['UNIDADE'] = "ppm"

    return df

for pol in lista_pols:

    print(pol)

    df_pol = df_est[df_est["Poluente"] == pol]

    lista_estacoes = list(set(df_pol['ID_MMA']))

    print(lista_estacoes)

    for estacao in lista_estacoes:

        if type(estacao) == str:

            cod_poluente = int(tabela_pols.loc[tabela_pols['POLUENTE'] == pol, 'COD_POLUENTE'].values[0])
           
            cod_poluente = f"{cod_poluente:03d}"

            print(cod_poluente)

            df_pol_est = df_pol[df_pol["ID_MMA"] == estacao]

            padrao = os.path.join(os.getcwd() + '/data/MQAr/'+pol+'/', f"{estacao}*.csv")
            arquivos = glob.glob(padrao)

            for arquivo in arquivos:

                print(arquivo)
                
                df_mma = pd.read_csv(arquivo, encoding = 'UTF-8')

                df_iema = df_pol_est[['Data','Hora','Valor','Unidade']]
                
                mask_24 = df_iema['Hora'].str.startswith('24')

                df_iema.loc[mask_24, 'Data'] = pd.to_datetime(df_iema.loc[mask_24, 'Data']) + pd.Timedelta(days=1)
                df_iema.loc[mask_24, 'Hora'] = '00:00:00'
                
                df_iema['Data'] = df_iema['Data'].astype(str)
                
                df_iema['DATETIME'] = pd.to_datetime(df_iema['Data'] + ' ' + df_iema['Hora'], errors='coerce')
                
                df_iema.drop(columns=['Data','Hora'], inplace=True)

                df_iema.rename(columns={'Valor':'VALOR_ORIGINAL','Unidade':'UNIDADE'},inplace=True)

                df_mma['DATETIME'] = pd.to_datetime(df_mma['DATETIME'], errors='coerce')
                df_iema['DATETIME'] = pd.to_datetime(df_iema['DATETIME'], errors='coerce')
                
                df_mma = df_mma.dropna(subset=['DATETIME'])
                df_iema = df_iema.dropna(subset=['DATETIME'])

                if pol == 'CO':
                    df_iema = ug_to_ppm(df_iema)
                
                df_iema = (
                    df_iema.sort_values('VALOR_ORIGINAL', ascending=False)
                           .drop_duplicates(subset='DATETIME', keep='first')
                )
                
                df_final = df_mma.merge(df_iema, on='DATETIME', how='left', suffixes=('', '_df2'))
                
                df_final['VALOR_ORIGINAL'] = df_final['VALOR_ORIGINAL_df2'].combine_first(df_final['VALOR_ORIGINAL'])
                df_final['UNIDADE'] = df_final['UNIDADE_df2'].combine_first(df_final['UNIDADE'])

                df_final['VALOR'] = df_final['VALOR_ORIGINAL']

                df_final.drop(columns=['VALOR_ORIGINAL'], inplace=True)

                df_final = create_QAQCMMA_VALOR(df_final,pol)
                
                print('iema')
                print(df_iema.head())
                print('mma')
                print(df_mma.head())
                print('final')
                print(df_final.head())

                df_final.to_csv(arquivo, index=False)


['FMC', 'CO', 'NO', 'PM10', 'MP10', 'O3', 'NO2', 'SO2', 'PTS', 'MP25']
FMC
['SP0099', 'SP0083', nan, 'SP0091']
006
006
006
CO
['MG0003', 'RJ0043', 'RJ0022', 'RJ0044', 'PR0009 ', 'MG0001', 'RJ0039', 'BA0011', 'RJ0225', 'SP0120', 'MG0006', 'SP0085', 'MG0008', 'RJ0070', 'RJ0226', 'MG0010', 'MG0017', 'RJ0018', 'RJ0023', 'RJ0038', 'MG0002', 'PR0014 ', 'PR0007 ', 'PR0006 ', 'RJ0079', 'RJ0029', 'PR0004 ', 'RJ0057', 'RJ0048', 'SP0064', 'MG0009', 'MG0057', 'SP0086', 'MG0053', 'RJ0027', 'RJ0036', 'RJ0078', 'SP0280', 'RJ0033', 'RJ0081', 'RJ0037', 'PR0001 ', 'SP0103', 'RJ0030', 'MG0012', 'PR0012 ', 'RJ0026', 'BA0008', 'SP0099', 'SP0263', 'MG0013', 'PR0011 ', 'MG0005', 'MG0004', 'RJ0031', 'BA0007', 'RJ0021', 'PR0015 ', 'RJ0019', 'SP0091', 'SP0073', 'BA0005', 'SP0288', 'RJ0058', 'RJ0071', 'RJ0045', 'CE0001', nan, 'MG0039', 'RJ0069', 'BA0003', 'PR0008 ', 'RJ0215', 'BA0010', 'RJ0055', 'SP0083']
007
/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/CO/MG0003RA007.csv


/tmp/ipykernel_273420/1352582357.py:97: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mma = pd.read_csv(arquivo, encoding = 'UTF-8')
/tmp/ipykernel_273420/1352582357.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['Data'] = df_iema['Data'].astype(str)
/tmp/ipykernel_273420/1352582357.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['DATETIME'] = pd.to_datetime(df_iema['Data'] + ' ' + df_iema['Hora'], errors='coerce')
/tmp/

iema
          VALOR_ORIGINAL UNIDADE            DATETIME
37205410           13.52     ppm 2021-01-06 11:30:00
37205568           11.31     ppm 2021-01-06 12:30:00
36610940            8.47     ppm 2018-07-06 09:30:00
32042527            7.78     ppm 2020-04-16 08:30:00
37919342            7.60     ppm 2021-07-14 18:30:00
mma
             DATETIME     ANO  MES  DIA  HORA  VALOR VALOR_ORIGINAL UNIDADE  \
0 2009-01-01 01:00:00  2009.0  1.0  1.0   1.0    0.3            0.3     ppm   
1 2009-01-01 02:00:00  2009.0  1.0  1.0   2.0    0.2            0.2     ppm   
2 2009-01-01 03:00:00  2009.0  1.0  1.0   3.0    0.2            0.2     ppm   
3 2009-01-01 04:00:00  2009.0  1.0  1.0   4.0    0.2            0.2     ppm   
4 2009-01-01 05:00:00  2009.0  1.0  1.0   5.0    0.2            0.2     ppm   

   QAQC_INTERNO  QAQC_MMA  
0          True      True  
1          True      True  
2          True      True  
3          True      True  
4          True      True  
final
             DATETIME   

/tmp/ipykernel_273420/1352582357.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['Data'] = df_iema['Data'].astype(str)
/tmp/ipykernel_273420/1352582357.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['DATETIME'] = pd.to_datetime(df_iema['Data'] + ' ' + df_iema['Hora'], errors='coerce')
/tmp/ipykernel_273420/1352582357.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/

iema
          VALOR_ORIGINAL UNIDADE            DATETIME
28736083     1721.144247     ppm 2019-10-02 05:00:00
28735901     1637.147108     ppm 2019-10-02 04:00:00
28736278      686.684452     ppm 2019-10-02 06:00:00
28735710      553.582357     ppm 2019-10-02 03:00:00
28735522      419.468408     ppm 2019-10-02 02:00:00
mma
             DATETIME     ANO  MES  DIA  HORA  VALOR  VALOR_ORIGINAL UNIDADE  \
0 2009-06-06 20:00:00  2009.0  6.0  6.0  20.0   0.24            0.24     ppm   
1 2009-06-06 21:00:00  2009.0  6.0  6.0  21.0   0.24            0.24     ppm   
2 2009-06-06 22:00:00  2009.0  6.0  6.0  22.0   0.24            0.24     ppm   
3 2009-06-06 23:00:00  2009.0  6.0  6.0  23.0   0.24            0.24     ppm   
4 2009-06-07 00:00:00  2009.0  6.0  7.0   0.0   0.24            0.24     ppm   

   QAQC_INTERNO  QAQC_MMA  
0          True      True  
1          True      True  
2          True      True  
3          True      True  
4          True      True  
final
             DATET

...

002
/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/MP25/RJ0069RA002.csv
iema
          VALOR_ORIGINAL UNIDADE            DATETIME
22841549          109.45   ug/m3 2022-09-02 09:00:00
24535538           94.00   µg/m³ 2021-09-21 07:00:00
24534821           91.13   µg/m³ 2021-08-22 08:00:00
22840281           90.63   ug/m3 2022-07-11 13:00:00
24533611           85.48   µg/m³ 2021-07-02 11:00:00
mma
             DATETIME   ANO  MES  DIA  HORA  VALOR  VALOR_ORIGINAL UNIDADE  \
0 2021-01-18 09:00:00  2021    1   18     9   2.29            2.29   µg/m³   
1 2021-01-18 10:00:00  2021    1   18    10   2.27            2.27   µg/m³   
2 2021-01-18 11:00:00  2021    1   18    11   1.85            1.85   µg/m³   
3 2021-01-18 12:00:00  2021    1   18    12   2.30            2.30   µg/m³   
4 2021-01-18 13:00:00  2021    1   18    13   0.75            0.75   µg/m³   

   QAQC_INTERNO  QAQC_MMA  
0          True      True  
1          True      True  
2          True      True  
3          True     

/tmp/ipykernel_273420/1352582357.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['Data'] = df_iema['Data'].astype(str)
/tmp/ipykernel_273420/1352582357.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['DATETIME'] = pd.to_datetime(df_iema['Data'] + ' ' + df_iema['Hora'], errors='coerce')
/tmp/ipykernel_273420/1352582357.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/

002
/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/MP25/SP0063RA002.csv


/tmp/ipykernel_273420/1352582357.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['Data'] = df_iema['Data'].astype(str)
/tmp/ipykernel_273420/1352582357.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['DATETIME'] = pd.to_datetime(df_iema['Data'] + ' ' + df_iema['Hora'], errors='coerce')
/tmp/ipykernel_273420/1352582357.py:110: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/

iema
          VALOR_ORIGINAL UNIDADE            DATETIME
10541008           234.0   ug/m3 2020-01-01 03:00:00
10541009           156.0   ug/m3 2020-01-01 04:00:00
16803772           135.0   ug/m3 2022-01-01 05:00:00
16808091           124.0   ug/m3 2022-07-24 08:00:00
7285021            122.0   ug/m3 2017-07-21 04:00:00
mma
             DATETIME   ANO  MES  DIA  HORA  VALOR  VALOR_ORIGINAL UNIDADE  \
0 2017-05-29 01:00:00  2017    5   29     1   42.0            42.0   µg/m3   
1 2017-05-29 02:00:00  2017    5   29     2   49.0            49.0   µg/m3   
2 2017-05-29 03:00:00  2017    5   29     3   42.0            42.0   µg/m3   
3 2017-05-29 04:00:00  2017    5   29     4   34.0            34.0   µg/m3   
4 2017-05-29 05:00:00  2017    5   29     5   26.0            26.0   µg/m3   

   QAQC_INTERNO  QAQC_MMA  
0          True      True  
1          True      True  
2          True      True  
3          True      True  
4          True      True  
final
             DATETIME   ANO  M

In [529]:
pol

'PM10'

In [527]:
df_final = df_mma.merge(df_iema, on='DATETIME', how='left', suffixes=('', '_df2'))

df_mma = pd.read_csv(arquivo, encoding = 'UTF-8')

df_iema = df_pol_est[['Data','Hora','Valor','Unidade']]

mask_24 = df_iema['Hora'].str.startswith('24')

df_iema.loc[mask_24, 'Data'] = pd.to_datetime(df_iema.loc[mask_24, 'Data']) + pd.Timedelta(days=1)
df_iema.loc[mask_24, 'Hora'] = '00:00:00'

df_iema['Data'] = df_iema['Data'].astype(str)

df_iema['DATETIME'] = pd.to_datetime(df_iema['Data'] + ' ' + df_iema['Hora'], errors='coerce')

df_iema.drop(columns=['Data','Hora'], inplace=True)

df_iema.rename(columns={'Valor':'VALOR_ORIGINAL','Unidade':'UNIDADE'},inplace=True)

df_mma['DATETIME'] = pd.to_datetime(df_mma['DATETIME'], errors='coerce')
df_iema['DATETIME'] = pd.to_datetime(df_iema['DATETIME'], errors='coerce')

df_mma = df_mma.dropna(subset=['DATETIME'])
df_iema = df_iema.dropna(subset=['DATETIME'])

if pol == 'CO':
    df_iema = ug_to_ppm(df_iema)

df_iema = (
    df_iema.sort_values('VALOR_ORIGINAL', ascending=False)
           .drop_duplicates(subset='DATETIME', keep='first')
)

df_final = df_mma.merge(df_iema, on='DATETIME', how='left', suffixes=('', '_df2'))


/tmp/ipykernel_175285/2951050035.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['Data'] = df_iema['Data'].astype(str)
/tmp/ipykernel_175285/2951050035.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_iema['DATETIME'] = pd.to_datetime(df_iema['Data'] + ' ' + df_iema['Hora'], errors='coerce')
/tmp/ipykernel_175285/2951050035.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pan

,DATETIME,ANO,MES,DIA,HORA,VALOR,QAQC_INTERNO,UNIDADE
0,1998-01-01 00:00:00,1998,1,1,0,NaN,0.0,ppm
1,1998-01-01 01:00:00,1998,1,1,1,NaN,0.0,ppm
2,1998-01-01 02:00:00,1998,1,1,2,NaN,0.0,ppm
3,1998-01-01 03:00:00,1998,1,1,3,NaN,0.0,ppm
4,1998-01-01 04:00:00,1998,1,1,4,NaN,0.0,ppm
...,...,...,...,...,...,...,...,...
236683,2024-12-31 19:00:00,2024,12,31,19,0.1,100.0,ppm
236684,2024-12-31 20:00:00,2024,12,31,20,0.1,100.0,ppm
236685,2024-12-31 21:00:00,2024,12,31,21,0.1,100.0,ppm
236686,2024-12-31 22:00:00,2024,12,31,22,0.1,100.0,ppm


In [ ]:
caminho = os.getcwd()+'/data/DADOS_BRUTOS/BRUTO_MONITORAR/'

df_colunas_monitorar = pd.read_csv(caminho+'colunas_monitorar.csv')

estados = os.listdir(caminho)

lista = []

lista_validacao = []

mapeamento = dict(zip(df_colunas_monitorar["COLUNAS_MONITORAR"], df_colunas_monitorar["COLUNAS_MMA"]))

dict_estacao_pol = {}

def unit_pol(pol):

    if pol in ['NOX','NO2','PTS','O3','MP10','MP25','NO','ERT','SO2']:
        unidade = 'µg/m³'
    elif pol in ['HCT','H2S','NH3','HCNM','BENZENO','CO','CH4']:
        unidade = 'ppm'

    return unidade
    
for estado in estados:

    print(estado)
    
    if '.csv' not in estado and estado != '.ipynb_checkpoints':

        if estado == 'RJ' or estado == 'PA' or estado == 'RN':
            df_rj_ou_pa_estacao = pd.read_csv(os.getcwd()+'/data/DADOS_BRUTOS/BRUTO_MONITORAR/'+estado+'_ID_MMA_monitorar.csv')
        else:
            df_estado_estacao = pd.read_csv(os.getcwd()+'/data/DADOS_ESTACOES/'+estado+'_estacoes.csv')

        estacoes = os.listdir(caminho+estado)
    
        for estacao in estacoes:
    
            if estacao != '.ipynb_checkpoints':
    
                df = pd.read_csv(caminho+estado+'/'+estacao, sep = ';')
                    
                df = df.rename(columns=mapeamento)

                df = df.dropna(axis=1, how='all')
                
                df_consolidado = df.groupby(df.columns, axis=1).first() 

                print(estacao)

                for col in df_consolidado.columns:
    
                    if col != 'DATA' and col != 'HORA' and 'QAQC' not in col:

                        df_consolidado[col] = df_consolidado[col].apply(
                            lambda x: str(x).replace('.', '') if pd.notna(x) else x
                        )
                        
                        df_consolidado[col] = df_consolidado[col].apply(
                            lambda x: str(x).replace(',', '.') if pd.notna(x) else x
                        ).astype(float)

                for pol in ['NOX','HCT','H2S','NO2','PTS','NH3','HCNM','HORA','O3','BENZENO','CO','MP10','MP25','CH4','NO','ERT','SO2']:
    
                    if pol in df_consolidado.columns and pol != 'HORA' and pol != 'DATA':
    
                        unidade = 'desconhecida'

                        print(pol)

                        qaqc = 'QAQC_'+pol
                        if qaqc in df_consolidado.columns:
                            print(df_consolidado.columns)
                            df_monitorar = df_consolidado[['DATA','HORA',pol,qaqc]]
                        else:
                            df_monitorar = df_consolidado[['DATA','HORA',pol]]

                        df_monitorar['DATA_HORA_STRING'] = df_monitorar['DATA'] + ' ' + df_monitorar['HORA']

                        df_monitorar['DATETIME'] = pd.to_datetime(
                            df_monitorar['DATA_HORA_STRING'],
                            format='%d/%m/%Y %H:%M'
                        )
                        
                        df_monitorar = df_monitorar.drop(columns=['DATA_HORA_STRING','DATA','HORA'])
    
                        df_monitorar.index = df_monitorar['DATETIME']
                        
                        df_monitorar.insert(1, 'ANO', df_monitorar.index.year)
                        df_monitorar.insert(2, 'MES', df_monitorar.index.month)
                        df_monitorar.insert(3, 'DIA', df_monitorar.index.day)
                        df_monitorar.insert(4, 'HORA', df_monitorar.index.hour)

                        df_monitorar['UNIDADE'] = unidade

                        dict_estacao_pol[estacao[:-4]+'_'+pol] = df_monitorar

                        if qaqc in df_monitorar.columns:
        
                            for validacao in list(set(df_monitorar[qaqc])):

                                lista_validacao.append(validacao)

                        print(estacao[:-4])
                        
                        if estado == 'RJ' or estado == 'PA' or estado == 'RN':
                            id_mma = df_rj_ou_pa_estacao.loc[df_rj_ou_pa_estacao['STATION']==estacao[:-4], 'ID_MMA'].iloc[0]
                        else:
                            mask = (df_estado_estacao['ID_OEMA']
                                .str.replace(' ', '', regex=False)
                                .str.replace('-', '', regex=False)
                                .str.lower()
                                == estacao[:-4].replace(' ', '').replace('-', '').lower()
                            )
                    
                            id_mma = df_estado_estacao.loc[mask, 'ID_MMA'].iloc[-1]
                            
                        print(id_mma)
                        
                        df_monitorar.rename(columns={pol:'VALOR_ORIGINAL','QAQC_'+pol:'QAQC_INTERNO'},inplace=True)

                        padrao = os.path.join(os.getcwd() + '/data/MQAr/'+pol+'/', f"{id_mma}*.csv")
                        arquivos = glob.glob(padrao)

                        if len(arquivos) == 0:

                            unidade = unit_pol(pol)

                            df_monitorar['UNIDADE'] = unidade
                
                            df_monitorar = create_QAQCMMA_VALOR(df_monitorar,pol)

                            cod_poluente = int(tabela_pols.loc[tabela_pols['POLUENTE'] == pol, 'COD_POLUENTE'].values[0])
                           
                            cod_poluente = f"{cod_poluente:03d}"

                            print(str(pol)+'/'+str(id_mma)+'ND'+cod_poluente+'.csv')
                            
                            df_monitorar.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+pol+'/'+id_mma+'ND'+cod_poluente+'.csv', index=False)
                            
                        else:
                            for arquivo in arquivos:
                    
                                df_mma = pd.read_csv(arquivo, encoding = 'UTF-8')
                                
                                df_mma = df_mma.dropna(subset=['DATETIME'])
                                df_monitorar = df_monitorar.dropna(subset=['DATETIME'])
                                
                                df_monitorar = (
                                    df_monitorar.sort_values('VALOR_ORIGINAL', ascending=False)
                                           .drop_duplicates(subset='DATETIME', keep='first')
                                )

                                df_monitorar = df_monitorar.reset_index(drop=True)

                                df_mma['DATETIME'] = pd.to_datetime(df_mma['DATETIME'], errors='coerce')
                                df_monitorar['DATETIME'] = pd.to_datetime(df_monitorar['DATETIME'], errors='coerce')

                                unidade = unit_pol(pol)

                                df_monitorar['UNIDADE'] = unidade
                                
                                df_final = df_mma.merge(df_monitorar, on='DATETIME', how='left', suffixes=('', '_df2'))
                                
                                df_final['VALOR_ORIGINAL'] = df_final['VALOR_ORIGINAL_df2'].combine_first(df_final['VALOR_ORIGINAL'])
                                df_final['UNIDADE'] = df_final['UNIDADE_df2'].combine_first(df_final['UNIDADE'])
                
                                df_final['VALOR'] = df_final['VALOR_ORIGINAL']
                
                                df_final.drop(columns=['VALOR_ORIGINAL','ANO','MES','DIA','HORA'], inplace=True)
                                df_final=df_final.set_index('DATETIME')
                    
                                lista_horas = pd.date_range(
                                    start=df_final.index.min(), 
                                    end=df_final.index.max(), 
                                    freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()

                                df_final = df_final[~df_final.index.duplicated(keep='first')]
                                
                                if len(lista_horas) != len(df_final):
                                    df_final = df_final.reindex(pd.DatetimeIndex(lista_horas))
                                
                                df_final['DATETIME'] = df_final.index
                                
                                cols = ["DATETIME"] + [c for c in df_final.columns if c != "DATETIME"]
                                df_final = df_final[cols]
                                
                                df_final.index = df_final['DATETIME']
                                
                                df_final.insert(1, 'ANO', df_final.index.year)
                                df_final.insert(2, 'MES', df_final.index.month)
                                df_final.insert(3, 'DIA', df_final.index.day)
                                df_final.insert(4, 'HORA', df_final.index.hour)
                
                                df_final = create_QAQCMMA_VALOR(df_final,pol)
                                
                                print('iema')
                                print(df_monitorar.head())
                                print('mma')
                                print(df_mma.head())
                                print('final')
                                print(df_final.head())
                                
                                df_final.to_csv(arquivo, index=False)
                            
                                                                
lista_colunas = list(set(lista))

lista_validacao = list(set(lista_validacao))

colunas_monitorar.csv
DF
FercalEscola.csv
MP10
Index(['DATA', 'HORA', 'MP10', 'MP25', 'QAQC_MP10', 'QAQC_MP25'], dtype='object')
FercalEscola
DF0002
iema
   VALOR_ORIGINAL   ANO  MES  DIA  HORA QAQC_INTERNO            DATETIME  \
0         323.084  2024   12    3     8           VA 2024-12-03 08:00:00   
1         294.205  2024   12    3     7           VA 2024-12-03 07:00:00   
2         218.271  2024    8   25    23           VA 2024-08-25 23:00:00   
3         204.446  2024    8   25    22           VA 2024-08-25 22:00:00   
4         180.625  2024   12    4     8           VA 2024-12-04 08:00:00   

  UNIDADE  
0   µg/m³  
1   µg/m³  
2   µg/m³  
3   µg/m³  
4   µg/m³  
mma
             DATETIME   ANO  MES  DIA  HORA  VALOR  VALOR_ORIGINAL UNIDADE  \
0 2024-01-01 01:00:00  2024    1    1     1  29.25           29.25   ug/m3   
1 2024-01-01 02:00:00  2024    1    1     2  26.62           26.62   ug/m3   
2 2024-01-01 03:00:00  2024    1    1     3  22.00           22.00   ug/m3   
3

/tmp/ipykernel_273420/3647521321.py:47: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_consolidado = df.groupby(df.columns, axis=1).first()
/tmp/ipykernel_273420/3647521321.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_monitorar['DATA_HORA_STRING'] = df_monitorar['DATA'] + ' ' + df_monitorar['HORA']
/tmp/ipykernel_273420/3647521321.py:173: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(


MP25
Index(['DATA', 'HORA', 'MP10', 'MP25', 'QAQC_MP10', 'QAQC_MP25'], dtype='object')
FercalEscola
DF0002
iema
   VALOR_ORIGINAL   ANO  MES  DIA  HORA QAQC_INTERNO            DATETIME  \
0         179.744  2024    8   25    23           VA 2024-08-25 23:00:00   
1         165.605  2024    8   25    22           VA 2024-08-25 22:00:00   
2         145.700  2024    8   26     0           VA 2024-08-26 00:00:00   
3         128.109  2024    8   26     1           VA 2024-08-26 01:00:00   
4         109.101  2024    8   26     2           VA 2024-08-26 02:00:00   

  UNIDADE  
0   µg/m³  
1   µg/m³  
2   µg/m³  
3   µg/m³  
4   µg/m³  
mma
             DATETIME   ANO  MES  DIA  HORA  VALOR  VALOR_ORIGINAL UNIDADE  \
0 2024-01-01 01:00:00  2024    1    1     1   9.87            9.87   ug/m3   
1 2024-01-01 02:00:00  2024    1    1     2   9.19            9.19   ug/m3   
2 2024-01-01 03:00:00  2024    1    1     3   8.07            8.07   ug/m3   
3 2024-01-01 04:00:00  2024    1    1     4

/tmp/ipykernel_273420/3647521321.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_monitorar['DATA_HORA_STRING'] = df_monitorar['DATA'] + ' ' + df_monitorar['HORA']
/tmp/ipykernel_273420/3647521321.py:173: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_273420/3647521321.py:47: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_consolidado = df.groupby(df.columns, axis=1).first()


FercalCRAS.csv
NO2
Index(['CO', 'DATA', 'HORA', 'MP10', 'MP25', 'NO2', 'O3', 'QAQC_CO',
       'QAQC_MP10', 'QAQC_MP25', 'QAQC_NO2', 'QAQC_O3', 'QAQC_SO2', 'SO2'],
      dtype='object')
FercalCRAS
DF0001
iema
   VALOR_ORIGINAL   ANO  MES  DIA  HORA QAQC_INTERNO            DATETIME  \
0           260.0  2024    8   24    10           VA 2024-08-24 10:00:00   
1           190.0  2024    8   24     9           VA 2024-08-24 09:00:00   
2           190.0  2024    9    5    10           VA 2024-09-05 10:00:00   
3           170.0  2024    8   23    22           VA 2024-08-23 22:00:00   
4           160.0  2024    8   23    21           VA 2024-08-23 21:00:00   

  UNIDADE  
0   µg/m³  
1   µg/m³  
2   µg/m³  
3   µg/m³  
4   µg/m³  
mma
             DATETIME   ANO  MES  DIA  HORA  VALOR  VALOR_ORIGINAL UNIDADE  \
0 2024-01-01 01:00:00  2024    1    1     1   3.50            3.50   ug/m3   
1 2024-01-01 02:00:00  2024    1    1     2   3.46            3.46   ug/m3   
2 2024-01-01 03:00:00  2

...

/tmp/ipykernel_273420/3647521321.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_monitorar['DATA_HORA_STRING'] = df_monitorar['DATA'] + ' ' + df_monitorar['HORA']
/tmp/ipykernel_273420/3647521321.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_monitorar['DATETIME'] = pd.to_datetime(
/tmp/ipykernel_273420/3647521321.py:144: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mma = pd.read_csv(arquivo, encoding = 'UTF-8')
/tmp/ipykernel_2734

iema
   VALOR_ORIGINAL   ANO  MES  DIA  HORA QAQC_INTERNO            DATETIME  \
0          9999.9  2022    5   12    14           IV 2022-05-12 14:00:00   
1          9999.9  2022    5   12     6           IV 2022-05-12 06:00:00   
2          9999.9  2022    5   12    13           IV 2022-05-12 13:00:00   
3          9999.9  2022    5   19    11           IV 2022-05-19 11:00:00   
4          9999.9  2022    5   12     9           IV 2022-05-12 09:00:00   

  UNIDADE  
0   µg/m³  
1   µg/m³  
2   µg/m³  
3   µg/m³  
4   µg/m³  
mma
             DATETIME     ANO  MES  DIA  HORA  VALOR VALOR_ORIGINAL UNIDADE  \
0 2009-01-01 01:00:00  2009.0  1.0  1.0   1.0   11.5           11.5   µg/m3   
1 2009-01-01 02:00:00  2009.0  1.0  1.0   2.0   52.2           52.2   µg/m3   
2 2009-01-01 03:00:00  2009.0  1.0  1.0   3.0   18.0             18   µg/m3   
3 2009-01-01 04:00:00  2009.0  1.0  1.0   4.0    NaN              0   µg/m3   
4 2009-01-01 05:00:00  2009.0  1.0  1.0   5.0    NaN              0

/tmp/ipykernel_273420/3647521321.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_monitorar['DATA_HORA_STRING'] = df_monitorar['DATA'] + ' ' + df_monitorar['HORA']
/tmp/ipykernel_273420/3647521321.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_monitorar['DATETIME'] = pd.to_datetime(
/tmp/ipykernel_273420/3647521321.py:144: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mma = pd.read_csv(arquivo, encoding = 'UTF-8')
/tmp/ipykernel_2734

iema
   VALOR_ORIGINAL   ANO  MES  DIA  HORA QAQC_INTERNO            DATETIME  \
0           985.0  2023    1   11    21           IM 2023-01-11 21:00:00   
1           985.0  2024   12   12     8           IM 2024-12-12 08:00:00   
2           985.0  2022    4   12    15           IM 2022-04-12 15:00:00   
3           985.0  2022    4   12    14           IM 2022-04-12 14:00:00   
4           985.0  2022    4   12    13           IM 2022-04-12 13:00:00   

  UNIDADE  
0   µg/m³  
1   µg/m³  
2   µg/m³  
3   µg/m³  
4   µg/m³  
mma
             DATETIME     ANO  MES  DIA  HORA  VALOR VALOR_ORIGINAL UNIDADE  \
0 2009-01-01 01:00:00  2009.0  1.0  1.0   1.0    NaN              *   µg/m3   
1 2009-01-01 02:00:00  2009.0  1.0  1.0   2.0    NaN              *   µg/m3   
2 2009-01-01 03:00:00  2009.0  1.0  1.0   3.0    NaN              *   µg/m3   
3 2009-01-01 04:00:00  2009.0  1.0  1.0   4.0    NaN              *   µg/m3   
4 2009-01-01 05:00:00  2009.0  1.0  1.0   5.0    NaN              *

/tmp/ipykernel_273420/3647521321.py:47: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_consolidado = df.groupby(df.columns, axis=1).first()


In [210]:
lista_validacao

['VA', None, 'IM', 'VU', 'ID', 'IF', 'VE', 'IC', 'IN', 'VR', 'IV', 'IO']

In [199]:
df_monitorar.columns

Index(['NO2', 'ANO', 'MES', 'DIA', 'HORA', 'DATETIME', 'UNIDADE'], dtype='object')

In [97]:
df_consolidado

linhas_encontradas = df_consolidado[df_consolidado[col] == '2.645,499']

linhas_encontradas

,DATA,HORA,MP10,MP25,PTS,QAQC_MP10,QAQC_MP25,QAQC_PTS
2677,23/04/2022,17:30,"2.645,499","2.257,856","2.575,189",IM,IV,IV


# Representatividade Temporal

In [1]:
import pandas as pd
import os
import numpy as np

os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

df_rep_temporal = pd.DataFrame({
    'POLUENTES': ['MP10','MP25','SO2','NO2','O3','FMC','CO','PTS','Pb'],
    'DIA': ['24','24','24','1','8','24','8','1',''],
    'MES': ['mensal','mensal','mensal','mensal','mensal','mensal','mensal','mensal_geom','mensal'],
    'ANO': ['anual','anual','anual','anual','anual','anual','anual','anual_geom','anual']
}).set_index("POLUENTES")

print(df_rep_temporal)

          DIA          MES         ANO
POLUENTES                             
MP10       24       mensal       anual
MP25       24       mensal       anual
SO2        24       mensal       anual
NO2         1       mensal       anual
O3          8       mensal       anual
FMC        24       mensal       anual
CO          8       mensal       anual
PTS         1  mensal_geom  anual_geom
Pb                  mensal       anual


In [3]:
for ano, dados in df_dia.groupby(['ANO']):
    print(ano)
    qntd_valor = dados['VALOR'].notna().sum()
    if (ano[0] % 4 == 0 and ano[0] % 100 != 0) or (ano[0] % 400 == 0):
        dias = 366
    else:
        dias = 365
    prcnt_rep = (100*qntd_valor/dias)
    print(prcnt_rep)

(2009,)
69.04109589041096
(2010,)
98.63013698630137
(2011,)
80.82191780821918
(2012,)
94.80874316939891
(2013,)
96.16438356164383
(2014,)
99.17808219178082
(2015,)
98.9041095890411
(2016,)
88.79781420765028
(2017,)
99.72602739726027
(2018,)
97.26027397260275
(2019,)
91.5068493150685
(2020,)
99.72677595628416
(2021,)
98.9041095890411
(2022,)
99.72602739726027
(2023,)
100.0
(2024,)
97.81420765027322
(2025,)
67.12328767123287


In [2]:
def rep_temp(df,agrupamento,criterio,periodo_ref):

    resultados = []
    
    for chave, dados in df.groupby(agrupamento):
        qntd_valor = dados['VALOR'].notna().sum()  

        if criterio == 'HORA':
            qntd_tempo = 24
        elif criterio == 'DIA':
            if chave[1] in [4,6,9,11]:
                qntd_tempo = 30
            elif chave[1] in [1,3,5,7,8,10,12]:
                qntd_tempo = 31
            else:
                if (chave[0] % 4 == 0 and chave[0] % 100 != 0) or (chave[0] % 400 == 0):
                    qntd_tempo = 29
                else:
                    qntd_tempo = 28
        
        if qntd_valor >= (2/3) * qntd_tempo:
            if periodo_ref == "8horas":
                media = dados["VALOR"].rolling(window=8, min_periods=1).mean().max()
            else:
                media = periodo_ref(dados["VALOR"])
            rep = True
        else:
            media = np.nan   
            rep = False

        prcnt = 100*qntd_valor/qntd_tempo

        resultados.append((*chave, media, rep, prcnt))

    return resultados

def conta_dias_quadrimestre(ano,quadrimestre):

    if quadrimestre == 1:
        if (ano % 4 == 0 and ano % 100 != 0) or (ano % 400 == 0):
            dias = 121
        else:
            dias = 120
    elif quadrimestre == 2:
        dias = 123
    else:
        dias = 122

    return dias

def rep_temp_ano(df,agrupamento,criterio):

    resultados = []
    
    for chave, dados in df.groupby(agrupamento):
        qntd_valor = dados['VALOR'].notna().sum()
        qntd_tempo = conta_dias_quadrimestre(chave[0],chave[1])         
        
        if qntd_valor >= (1/2) * qntd_tempo:
            rep = True
        else:
            rep = False
    
        resultados.append((*chave, rep))

    return resultados
    
df_estacoes_rep_temporal = pd.DataFrame({
                    'ID_MMA_COMPLETO':[],
                    'PRCNT_REP_DIA':[],
                    'PRCNT_REP_MES':[],
                    'PRCNT_REP_ANO':[]
                })
                
for pol in df_rep_temporal.index:
    
    path = os.getcwd()+'/data/MQAr/' + pol + '/'

    print(pol)
    
    if os.path.isdir(path) and os.listdir(path):
        
        arquivos = os.listdir(path)
    
        for estacao in arquivos:

            if estacao.endswith('.csv'):

                print(estacao)

                df = pd.read_csv(path+estacao)

                df["VALOR"] = pd.to_numeric(df["VALOR"], errors="coerce")

                resultados_24 = rep_temp(df,['ANO','MES','DIA'],'HORA',np.mean)

                df_24 = pd.DataFrame(resultados_24, columns=['ANO', 'MES', 'DIA', 'VALOR', 'REP_DIA','PRCNT_REP'])

                df_24['DATETIME'] = pd.to_datetime(
                        dict(year=df_24["ANO"], month=df_24["MES"], day=df_24["DIA"])
                    )
            
                df_24 = df_24[['DATETIME','ANO','MES','DIA','VALOR','REP_DIA','PRCNT_REP']]

                if df_rep_temporal['DIA'][pol] == 'dia':
                    df_dia = df_24
                elif df_rep_temporal['DIA'][pol] == '8horas':
                    resultados_dia = rep_temp(df,['ANO','MES','DIA'],'HORA','8horas')

                    df_dia = pd.DataFrame(resultados_dia, columns=['ANO', 'MES', 'DIA', 'VALOR', 'REP_DIA','PRCNT_REP'])

                    df_dia['DATETIME'] = pd.to_datetime(dict(year=df_dia["ANO"], month=df_dia["MES"], day=df_dia["DIA"]))

                    df_dia = df_dia[['DATETIME','ANO','MES','DIA','VALOR','REP_DIA','PRCNT_REP']]
                        
                else:
                    resultados_dia = rep_temp(df,['ANO','MES','DIA'],'HORA',np.max)

                    df_dia = pd.DataFrame(resultados_dia, columns=['ANO', 'MES', 'DIA', 'VALOR', 'REP_DIA','PRCNT_REP'])

                    df_dia['DATETIME'] = pd.to_datetime(dict(year=df_dia["ANO"], month=df_dia["MES"], day=df_dia["DIA"]))

                    df_dia = df_dia[['DATETIME','ANO','MES','DIA','VALOR','REP_DIA','PRCNT_REP']]

                df_dia.to_csv(os.getcwd()+'/data/MQAr_averages/'+df_rep_temporal['DIA'][pol]+'/'+pol+'/'+estacao,index=False)

                resultados_mes = rep_temp(df_24,['ANO','MES'],'DIA',np.mean)

                df_mes = pd.DataFrame(resultados_mes, columns=['ANO', 'MES', 'VALOR', 'REP_MES','PRCNT_REP'])

                df_mes['DATETIME'] = pd.to_datetime(dict(year=df_mes["ANO"], month=df_mes["MES"], day=1))
            
                df_mes = df_mes[['DATETIME','ANO','MES','VALOR','REP_MES','PRCNT_REP']]

                df_mes.to_csv(os.getcwd()+'/data/MQAr_averages/'+df_rep_temporal['MES'][pol][:6]+'/'+pol+'/'+estacao,index=False)

                condicoes = [
                    (df_dia['MES'] <= 4),
                    (df_dia['MES'] >= 5) & (df_dia['MES'] <= 8),
                    (df_dia['MES'] >= 9)
                ]
                
                quadrimestre = [1, 2, 3]
                
                df_dia['QUADRIMESTRE'] = np.select(condicoes, quadrimestre)

                resultados_quad = rep_temp_ano(df_dia,['ANO','QUADRIMESTRE'],'QUADRIMESTRE')

                df_quad = pd.DataFrame(resultados_quad, columns=['ANO', 'QUADRIMESTRE', 'REP_QUAD'])
                
                df_ano_quad = df_quad.groupby("ANO", as_index=False).agg({"REP_QUAD": lambda x: x.sum() == 3})

                resultados = []
                
                for ano, dados in df_dia.groupby(['ANO']):
                    qntd_valor = dados['VALOR'].notna().sum()
                    if (ano[0] % 4 == 0 and ano[0] % 100 != 0) or (ano[0] % 400 == 0):
                        dias = 366
                    else:
                        dias = 365
                    prcnt_rep = (100*qntd_valor/dias)
                    if df_ano_quad.loc[df_ano_quad["ANO"] == ano[0], "REP_QUAD"].values[0] == True:
                        media = dados['VALOR'].mean()
                        rep = True
                    else:
                        media = np.nan
                        rep = False
                    resultados.append((*ano, media, rep, prcnt_rep))
                
                df_ano = pd.DataFrame(resultados, columns=['ANO', 'VALOR','REP_ANO','PRCNT_REP'])
                
                df_ano['DATETIME'] = pd.to_datetime(dict(year=df_ano["ANO"], month=1, day=1))
            
                df_ano = df_ano[['DATETIME','ANO','VALOR','REP_ANO','PRCNT_REP']]

                df_ano.to_csv(os.getcwd()+'/data/MQAr_averages/'+df_rep_temporal['ANO'][pol][:5]+'/'+pol+'/'+estacao, index=False)

                prcnt_dia = 100 * df_dia['VALOR'].notna().sum() / len(df_dia)
                prcnt_mes = 100 * df_mes['VALOR'].notna().sum() / len(df_mes)
                prcnt_ano = 100 * df_ano['VALOR'].notna().sum() / len(df_ano)

                prcnt_estacao = {'ID_MMA_COMPLETO': estacao[:-4], 'PRCNT_REP_DIA': prcnt_dia, 'PRCNT_REP_MES': prcnt_mes, 'PRCNT_REP_ANO': prcnt_ano}

                df_estacoes_rep_temporal = pd.concat([df_estacoes_rep_temporal, pd.DataFrame([prcnt_estacao])], ignore_index=True)

df_estacoes_rep_temporal.to_csv(os.getcwd()+'/data/MQAr_averages/REP_TEMPORAL.csv', index=False)
                
                

MP10
SP0248RA001.csv
RJ0031RA001.csv


KeyboardInterrupt: 

In [70]:
df_estacoes_rep_temporal

,ID_MMA_COMPLETO,PRCNT_REP_DIA,PRCNT_REP_MES,PRCNT_REP_ANO
0,SP0248RA001,96.243112,95.959596,88.235294
1,RJ0031RA001,88.713704,90.800000,80.952381
2,RJ0282RA001,84.054054,76.923077,0.000000


In [71]:
estacao

'SP0085RA001.csv'

In [74]:
condicoes = [
    (df_dia['MES'] <= 4),
    (df_dia['MES'] >= 5) & (df_dia['MES'] <= 8),
    (df_dia['MES'] >= 9)
]

quadrimestre = [1, 2, 3]

df_dia['QUADRIMESTRE'] = np.select(condicoes, quadrimestre)

resultados_quad = rep_temp_ano(df_dia,['ANO','QUADRIMESTRE'],'QUADRIMESTRE')

df_quad = pd.DataFrame(resultados_quad, columns=['ANO', 'QUADRIMESTRE', 'REP_QUAD'])

df_ano_quad = df_quad.groupby("ANO", as_index=False).agg({"REP_QUAD": lambda x: x.sum() == 3})
                

In [83]:
df_ano

,ANO,VALOR,REP_ANO
0,"(2009,)",NaN,False
1,"(2010,)",76.227778,True
2,"(2011,)",75.494915,True
3,"(2012,)",77.752161,True
4,"(2013,)",69.000000,True
5,"(2014,)",89.419890,True
6,"(2015,)",67.670360,True
7,"(2016,)",73.575385,True
8,"(2017,)",72.285714,True
9,"(2018,)",67.177465,True


In [41]:
df_pivot = df_mes.pivot(index='ANO', columns='MES', values='PRCNT_REP')

df_pivot = df_pivot.reindex(sorted(df_pivot.columns), axis=1)

df_pivot = df_pivot.reset_index()

df_pivot

MES,ANO,1,2,3,4,5,6,7,8,9,10,11,12
0,2009,NaN,NaN,NaN,53.333333,100.000000,100.000000,100.000000,87.096774,86.666667,100.000000,96.666667,100.000000
1,2010,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,96.666667,87.096774,100.000000,100.000000
2,2011,100.000000,100.000000,100.000000,100.000000,80.645161,0.000000,41.935484,100.000000,100.000000,96.774194,83.333333,67.741935
3,2012,58.064516,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,80.645161
4,2013,96.774194,96.428571,80.645161,100.000000,100.000000,93.333333,93.548387,100.000000,100.000000,93.548387,100.000000,100.000000
5,2014,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.322581,100.000000,100.000000,100.000000,100.000000
6,2015,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.000000,96.774194
7,2016,77.419355,100.000000,100.000000,100.000000,100.000000,86.666667,54.838710,100.000000,100.000000,100.000000,93.333333,54.838710
8,2017,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,96.774194,100.000000,100.000000
9,2018,100.000000,100.000000,100.000000,66.666667,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [9]:
df = pd.read_csv(os.getcwd()+'/data/MQAr/MP10/ES0001RA001.csv')

def create_QAQCMMA_VALOR(df,pol):

    df = df.rename(columns={'VALOR':'VALOR_ORIGINAL'})

    flags_invalidos = ['!', 'IF', 'IO', 'IC', 'I%', 'IL', 'IE', 'IS', 'IU', 'IM', 'IP', 'ID', 'IT', 'IR', 
                       'Fora da Faixa de Medição', 'Disabilitada Temporariamente', 'Inválido', 
                       'Insuficientes', 'Inexistente']

    df['QAQC_INTERNO'] = ~df['QAQC_INTERNO'].isin(flags_invalidos)
    
    DEFAULT_RANGE_LIMITS = {
        "O3": (0, 500),
        "CO": (0, 50),
        "NO2": (0, 1000),
        "NOX": (0, 2000),
        "SO2": (0, 1000),
        "MP25": (0, 1000),
        "MP10": (0, 2000),
    }

    df['QAQC_MMA'] = df['QAQC_INTERNO']

    if pol in list(DEFAULT_RANGE_LIMITS.keys()):
        lim_min = DEFAULT_RANGE_LIMITS[pol][0]
        lim_max = DEFAULT_RANGE_LIMITS[pol][1]
    else:
        lim_min = 0
        lim_max = np.inf

    df['VALOR'] = df['VALOR_ORIGINAL']

    df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce')
    
    df.loc[df['QAQC_MMA'] & (df['VALOR'].isna() | (df['VALOR'] < lim_min) | (df['VALOR'] > lim_max)), 'QAQC_MMA'] = False
    
    df.loc[~df['QAQC_MMA'], 'VALOR'] = np.nan
    
    df = df[['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR', 'VALOR_ORIGINAL', 'UNIDADE', 'QAQC_INTERNO', 'QAQC_MMA']]

    return df

    
    

In [15]:
DEFAULT_RANGE_LIMITS = {
        "O3": (0, 500),
        "CO": (0, 50),
        "NO2": (0, 1000),
        "NOX": (0, 2000),
        "SO2": (0, 1000),
        "MP25": (0, 1000),
        "MP10": (0, 2000),
    }

DEFAULT_RANGE_LIMITS['MP10'][1]

2000

In [17]:
list(DEFAULT_RANGE_LIMITS.keys())

['O3', 'CO', 'NO2', 'NOX', 'SO2', 'MP25', 'MP10']

In [29]:
pol = 'CCCC'

df['QAQC_MMA'] = df['QAQC_INTERNO']

if pol in list(DEFAULT_RANGE_LIMITS.keys()):
    lim_min = DEFAULT_RANGE_LIMITS[pol][0]
    lim_max = DEFAULT_RANGE_LIMITS[pol][1]
else:
    lim_min = 0
    lim_max = np.inf

In [30]:
df['VALOR'] = df['VALOR_ORIGINAL']

df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce')

df.loc[df['QAQC_MMA'] & (df['VALOR'].isna() | (df['VALOR'] < lim_min) | (df['VALOR'] > lim_max)), 'QAQC_MMA'] = False

df.loc[~df['QAQC_MMA'], 'VALOR'] = np.nan

df = df[['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR', 'VALOR_ORIGINAL', 'UNIDADE', 'QAQC_INTERNO', 'QAQC_MMA']]


In [32]:
df.columns

Index(['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR_ORIGINAL', 'UNIDADE',
       'QAQC_INTERNO', 'QAQC_MMA', 'VALOR'],
      dtype='object')

In [36]:
import pandas as pd
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

def ppb_to_ug(df,pol):

    if pol == 'so2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 2661260.49/10**6        

    elif pol == 'no2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1911038.92/10**6        

    elif pol == 'o3':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1993889.17/10**6        
    
    df.loc[:, "Unidade"] = "ug/m3"

    return df

def rectify_MT(path):

    dict_pols_stat = defaultdict(list)

    files = os.listdir(path)
    
    print(files)
    
    for item in files:
        
        estacao = " ".join(item.split('-')[1].split('.')[0].split('_')[0:2])
    
        print(estacao)
    
        df = pd.read_excel(path+item)
    
        df = df.drop(columns=['Nome da estação'])
    
        lista_pols = set(df['Poluente'])
    
        for pol in lista_pols:
    
            df_pol = df[df["Poluente"] == pol]
            
            if pol in ['no2','so2','o3']:
    
                df_pol = ppb_to_ug(df_pol,pol)
    
            df_pol_hora = df_pol.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"])
            
            df_pol_hora = df_pol_hora.filter(lambda g: len(g) >= 9)
            
            df_pol = (
                df_pol_hora.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"], as_index=False)
                      .agg({"Valor": "mean"})
            )
    
            df_pol['QAQC_INTERNO'] = None
    
            df_pol = df_pol.rename(columns={'Ano':'ANO',
                                            'Mes':'MES',
                                            'Dia':'DIA',
                                            'Hora':'HORA',
                                            'Unidade':'UNIDADE',
                                            'Valor':'VALOR'})
    
            for col in ["ANO", "MES", "DIA", "HORA"]:
                df_pol[col] = pd.to_numeric(df_pol[col], errors="coerce").astype("Int64")
            
            dict_pols_stat[estacao+'_'+pol].append(df_pol)

    dict_pols_MT = {
        'co':'CO',
        'no2': 'NO2',
        'so2': 'SO2',
        'o3': 'O3',
        'pm2p5':'MP25',
        'pm10': 'MP10'
    }

    dict_formatado = {}
    
    for chave in dict_pols_stat.keys():
        
        lista_dfs = dict_pols_stat[chave]
        
        df = pd.concat(lista_dfs, ignore_index=True)
    
        df["DATETIME"] = pd.to_datetime(
            df.apply(lambda r: f"{r.ANO}-{r.MES}-{r.DIA} {r.HORA}:00:00", axis=1)
        )
        df = df.set_index("DATETIME")
    
        df = df.sort_index()
        
        lista_horas = pd.date_range(
            start=df.index.min(), 
            end=df.index.max(), 
            freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
        
        if len(lista_horas) != len(df):
            df = df.reindex(pd.DatetimeIndex(lista_horas))
    
        df['DATETIME'] = df.index
    
        df = df[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']]
        
        dict_formatado[chave] = df
        
        primeiros_valores = {}
    
    for chave, df in dict_formatado.items():
        
        if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
            
            linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
            primeiros_valores[chave] = linha_valida["DATETIME"]
    
    codigo_estacao_MT = {}
    
    for chave in primeiros_valores.keys():
        
        station = chave.split('_')[0]
        data = primeiros_valores[chave]
        
        if station in codigo_estacao_MT:
            if data <= codigo_estacao_MT[station]:
                codigo_estacao_MT[station] = data
        else:
            codigo_estacao_MT[station] = data
    
    sorted_items = sorted(
        codigo_estacao_MT.items(),
        key=lambda x: (x[1], x[0])
    )
    
    codigo_estacao_MT = {}
    for i, (nome, ts) in enumerate(sorted_items, start=1):
        codigo = f"MT{i:04d}"
        codigo_estacao_MT[nome] = codigo
        
    for chave, df in dict_formatado.items():
        
        estacao = codigo_estacao_MT[chave.split('_')[0]]
        
        cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == dict_pols_MT[chave.split('_')[-1]], 'COD_POLUENTE'].values[0]
        
        nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]

        df = create_QAQCMMA_VALOR(df,nome_pasta)
        
        df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'IA'+str(cod_pol).zfill(3)+'.csv',index=False)

    dict_stations_MT = {
        'Sema':'CPA - SEMA - CBA',
        'BEA CBA': 'Dom Aquino - BEA - CBA',
        'CBM VG': 'Água Limpa - CBM - VG',
        'Mae Bonifacia': 'Duque de Caxias - Pq Mãe Bonifácia - CBA',
        'UFMT':'Boa Esperança - UFMT - CBA'
    }
    
    df_ids = pd.DataFrame({
        'ID_OEMA': codigo_estacao_MT.keys(),
        'ID_MMA':list(codigo_estacao_MT.values())})
    
    df_ids["ID_OEMA"] = df_ids["ID_OEMA"].replace(dict_stations_MT)

    print(df_ids)
    
    return df_ids

funcoes = {
    'MT': rectify_MT
}

#lista_estados = ['SC','RS','MT','DF','MA','BA','ES','MG','SP','RJ']
lista_estados = ['MT']

tabela_ids = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv')
tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')

for estado in lista_estados:
    
    path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

    df_ids = funcoes[estado](path)
    
    create_df_estacao(estado,df_ids)

['dados_monitoramento-Sema.xlsx', 'dados_monitoramento-BEA_CBA_24-25.xlsx', 'dados_monitoramento-BEA_CBA_22-23.xlsx', 'dados_monitoramento-CBM_VG_22-23.xlsx', 'dados_monitoramento-CBM_VG_24-25.xlsx', 'dados_monitoramento-Mae_Bonifacia_22-23.xlsx', 'dados_monitoramento-Mae_Bonifacia_24-25.xlsx', 'dados_monitoramento-UFMT.xlsx']
Sema
BEA CBA


KeyboardInterrupt: 

In [38]:
df = df_dia.groupby(['ANO'])

NameError: name 'df_dia' is not defined

In [72]:

tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')

In [73]:
for pol in tabela_pols['NOME_PASTA'].unique():
    print(pol)

MP10
MP25
SO2
NO2
O3
FMC
CO
PTS
CHUMBO
BENZENO
CH4
ERT
ETILBENZENO
H2S
NH3
HCNM
NO
NOX
MP1 
ACETAL
FORMAL
TOLUENO
XILENO
VOC
OXILENO
HCT
MPXILENO


In [68]:
import pandas as pd
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [18]:
funcoes = {
    
    'MT': rectify_MT,
}

lista_estados = ['MT']

tabela_ids = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv')
tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')


In [72]:

estado = 'DF'
  
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

#df_ids = funcoes[estado](path)
    
#create_df_estacao(estado,df_ids)

In [52]:
dict_pols_stat = defaultdict(list)

files = os.listdir(path)

print(files)

for item in files:
    
    estacao = " ".join(item.split('-')[1].split('.')[0].split('_')[0:2])

    print(estacao)

    df = pd.read_excel(path+item)

    df = df.drop(columns=['Nome da estação'])

    lista_pols = set(df['Poluente'])

    for pol in lista_pols:

        df_pol = df[df["Poluente"] == pol]
        
        if pol in ['no2','so2','o3']:

            df_pol = ppb_to_ug(df_pol,pol)

        df_pol_hora = df_pol.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"])
        
        df_pol_hora = df_pol_hora.filter(lambda g: len(g) >= 9)
        
        df_pol = (
            df_pol_hora.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"], as_index=False)
                  .agg({"Valor": "mean"})
        )

        df_pol['QAQC_INTERNO'] = None

        df_pol = df_pol.rename(columns={'Ano':'ANO',
                                        'Mes':'MES',
                                        'Dia':'DIA',
                                        'Hora':'HORA',
                                        'Unidade':'UNIDADE',
                                        'Valor':'VALOR'})

        for col in ["ANO", "MES", "DIA", "HORA"]:
            df_pol[col] = pd.to_numeric(df_pol[col], errors="coerce").astype("Int64")
        
        dict_pols_stat[estacao+'_'+pol].append(df_pol)



['dados_monitoramento-Sema.xlsx', 'dados_monitoramento-BEA_CBA_24-25.xlsx', 'dados_monitoramento-BEA_CBA_22-23.xlsx', 'dados_monitoramento-CBM_VG_22-23.xlsx', 'dados_monitoramento-CBM_VG_24-25.xlsx', 'dados_monitoramento-Mae_Bonifacia_22-23.xlsx', 'dados_monitoramento-Mae_Bonifacia_24-25.xlsx', 'dados_monitoramento-UFMT.xlsx']
Sema
BEA CBA
BEA CBA
CBM VG
CBM VG
Mae Bonifacia
Mae Bonifacia
UFMT


In [46]:
def ppb_to_ug(df,pol):

    if pol == 'so2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 2661260.49/10**6        

    elif pol == 'no2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1911038.92/10**6        

    elif pol == 'o3':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1993889.17/10**6        
    
    df.loc[:, "Unidade"] = "ug/m3"

    return df

In [60]:
dict_pols_MT = {
    'co':'CO',
    'no2': 'NO2',
    'so2': 'SO2',
    'o3': 'O3',
    'pm2p5':'MP25',
    'pm10': 'MP10'
}

dict_formatado = {}

for chave in dict_pols_stat.keys():
    
    lista_dfs = dict_pols_stat[chave]
    
    df = pd.concat(lista_dfs, ignore_index=True)

    df["DATETIME"] = pd.to_datetime(
        df.apply(lambda r: f"{r.ANO}-{r.MES}-{r.DIA} {r.HORA}:00:00", axis=1)
    )
    df = df.set_index("DATETIME")

    df = df.sort_index()
    
    lista_horas = pd.date_range(
        start=df.index.min(), 
        end=df.index.max(), 
        freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
    
    if len(lista_horas) != len(df):
        df = df.reindex(pd.DatetimeIndex(lista_horas))

    df['DATETIME'] = df.index

    df = df[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']]
    
    dict_formatado[chave] = df

primeiros_valores = {}

for chave, df in dict_formatado.items():
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_MT = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_MT:
        if data <= codigo_estacao_MT[station]:
            codigo_estacao_MT[station] = data
    else:
        codigo_estacao_MT[station] = data

sorted_items = sorted(
    codigo_estacao_MT.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_MT = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"MT{i:04d}"
    codigo_estacao_MT[nome] = codigo
    
for chave, df in dict_formatado.items():
    
    estacao = codigo_estacao_MT[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == dict_pols_MT[chave.split('_')[-1]], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})

return df_ids

/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = p

SyntaxError: 'return' outside function (2479180702.py, line 85)

In [61]:
df_ids

,ID_OEMA,ID_MMA
0,Mae Bonifacia,MT0001
1,BEA CBA,MT0002
2,CBM VG,MT0003
3,Sema,MT0004
4,UFMT,MT0005


In [185]:
from datetime import timedelta

def fix_24h(row):
    if isinstance(row, str) and row.startswith("24:"):
        # substitui 24: por 00:
        new_str = row.replace("24:", "00:", 1)
        # converte para datetime
        dt = pd.to_datetime(new_str, errors="coerce")
        # adiciona 1 dia
        if pd.notna(dt):
            dt += timedelta(days=1)
        return dt
    else:
        return pd.to_datetime(row, errors="coerce")



In [70]:
dict_stations_MT = {
        'Sema':'CPA - SEMA - CBA',
        'BEA CBA': 'Dom Aquino - BEA - CBA',
        'CBM VG': 'Água Limpa - CBM - VG',
        'Mae Bonifacia': 'Duque de Caxias - Pq Mãe Bonifácia - CBA',
        'UFMT':'Boa Esperança - UFMT - CBA'
    }
    
df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})


df_ids["ID_OEMA"] = df_ids["ID_OEMA"].replace(dict_stations_MT)


df_ids['ID_OEMA']

0    Duque de Caxias - Pq Mãe Bonifácia - CBA
1                      Dom Aquino - BEA - CBA
2                       Água Limpa - CBM - VG
3                            CPA - SEMA - CBA
4                  Boa Esperança - UFMT - CBA
Name: ID_OEMA, dtype: object

In [71]:
df_ids

,ID_OEMA,ID_MMA
0,Duque de Caxias - Pq Mãe Bonifácia - CBA,MT0001
1,Dom Aquino - BEA - CBA,MT0002
2,Água Limpa - CBM - VG,MT0003
3,CPA - SEMA - CBA,MT0004
4,Boa Esperança - UFMT - CBA,MT0005


In [120]:
estado = 'DF'

In [249]:
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

path = path + 'Monitor Report 2024_FINAL.xlsx'

df = pd.read_excel(path)

df.iloc[1] = df.iloc[1].ffill()

poluentes = ['CO_ppm','NO2_ug/m3','NO_ug/m3','NOx_ug/m3','O3_ug/m3','PM10','PM25','PTS','SO2_ug/m3']

dict_pols = {'CO_ppm':'CO',
             'NO2_ug/m3':'NO2',
             'NO_ug/m3':'NO',
             'NOx_ug/m3':'NOX',
             'O3_ug/m3':'O3',
             'PM10':'MP10',
             'PM25':'MP25',
             'PTS':'PTS',
             'SO2_ug/m3':'SO2'}

df.columns = df.iloc[1]

df = df.drop(index=[0, 1]).reset_index(drop=True)

df = df.rename(columns={'Date Time':'DATETIME'})

estacoes = set(df.columns[1:])

dict_pols_stat = defaultdict(list)

for estacao in estacoes:

    df_estacao = df[["DATETIME",estacao]]

    df_estacao.columns = [df_estacao.columns.tolist()[0]] + df_estacao.iloc[0, 1:].tolist()

    df_estacao = df_estacao.drop(index=[0]).reset_index(drop=True)

    for pol in poluentes:

        if pol in df_estacao.columns:
            
            df_pol = df_estacao[["DATETIME",pol]]

            df_pol['UNIDADE'] = df_pol[pol][0]

            df_pol = df_pol.drop(index=[0]).reset_index(drop=True)

            df_pol = df_pol[df_pol["DATETIME"].astype(str).str.contains(r"\d", na=False)].reset_index(drop=True)

            df_pol['DATETIME'] = df_pol['DATETIME'].apply(fix_24h)

            df_pol = df_pol.rename(columns={pol:'VALOR'})

            df_pol.index = df_pol['DATETIME']

            lista_horas = pd.date_range(
                start=df_pol.index.min(), 
                end=df_pol.index.max(), 
                freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
            
            if len(lista_horas) != len(df_pol):
                df_pol = df_pol.reindex(pd.DatetimeIndex(lista_horas))
            
            df_pol['QAQC_INTERNO'] = None
            
            df_pol.insert(1, 'ANO', df_pol.index.year)
            df_pol.insert(2, 'MES', df_pol.index.month)
            df_pol.insert(3, 'DIA', df_pol.index.day)
            df_pol.insert(4, 'HORA', df_pol.index.hour)

            pol = dict_pols[pol]

            df_pol['VALOR'] = pd.to_numeric(df_pol['VALOR'], errors='coerce')

            df_pol = df_pol[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']] 

            dict_pols_stat[estacao+'_'+pol] = df_pol
            

/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pol['UNIDADE'] = df_pol[pol][0]
/tmp/ipykernel_564959/1508039512.py:14: UserWarning: Parsing dates in %d:%M %m/%H/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(row, errors="coerce")
/tmp/ipykernel_564959/901552261.py:57: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

In [258]:
primeiros_valores = {}

for chave, df in dict_pols_stat.items():  
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_DF = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_DF:
        if data <= codigo_estacao_DF[station]:
            codigo_estacao_DF[station] = data
    else:
        codigo_estacao_DF[station] = data

sorted_items = sorted(
    codigo_estacao_DF.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_DF = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"DF{i:04d}"
    codigo_estacao_DF[nome] = codigo
    
for chave, df in dict_pols_stat.items():
    
    estacao = codigo_estacao_DF[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == chave.split('_')[-1], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

dict_oemas = {
    'Estação CRAS FERCAL': 'Fercal CRAS',
    'Estação Escola':	   'Fercal Escola'}

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_DF.keys(),
    'ID_MMA':list(codigo_estacao_DF.values())})

df_ids['ID_OEMA'] = df_ids['ID_OEMA'].replace(dict_oema)

return df_ids

In [275]:
df_ufs = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/IBGE_UFS_CODIGOS.csv')
    
cod_uf =  df_ufs.loc[df_ufs['UF'] == uf, 'CODIGOS'].values[0]

print(cod_uf)

if os.path.exists('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv'):

    df_estacao = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv')

    df_estacao['ID_MMA'] = df_estacao['ID_OEMA'].map(df_ids.set_index('ID_OEMA')['ID_MMA'])

else:

    colunas = ['ID_OEMA', 'UF', 'ID_MMA', 'COD_UF_IBGE', 'CIDADE', 'CD_MUN',
               'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE', 'LATITUDE',
               'LONGITUDE', 'MOBILIDADE', 'REALOCACAO', 'MARCA', 'CATEGORIA',
               'FUNCIONAMENTO', 'METODO', 'FINALIDADE', 'POLUENTE',
               'INICIO', 'STATUS', 'FIM', 'CALIBRACAO', 'OBS_CALIBRACAO', 'MONITORAR',
               'FONTE', 'OBS_GERAIS','DADOS_MONITORAMENTO','RECONHECIDA','REP_ESPACIAL_DECLARADA']
    
    df_estacao = pd.DataFrame(columns=colunas)

df_ids = pol_to_station(df_ids)

mapa = dict(zip(df_ids['ID_MMA'], df_ids['POLUENTE']))

df_estacao['POLUENTE'] = df_estacao['ID_MMA'].map(mapa).fillna(df_estacao['POLUENTE'])

#df_estacao = df_estacao.reindex(df_ids.index)

#df_estacao[["ID_MMA", "ID_OEMA", "POLUENTE"]] = df_ids[["ID_MMA", "ID_OEMA", "POLUENTE"]].values

df_estacao.loc[:, "COD_UF_IBGE"] = cod_uf
df_estacao.loc[:, "UF"] = uf
    
#df_estacao.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes_teste.csv', index=False)

53


In [276]:
df_estacao['POLUENTE']

0                                 PM2,5
1                                  PM10
2                             MP10,MP25
3                                  PM10
4                                  PM10
5                           MP2,5, MP10
6                           MP2,5, MP10
7                           MP2,5, MP10
8    MP10,NO,CO,PTS,O3,SO2,NO2,NOX,MP25
Name: POLUENTE, dtype: object

In [262]:
def pol_to_station(df_ids):

    base_path = Path('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/')

    id_to_poluentes = {}
    
    for poluente_dir in base_path.iterdir():
        if poluente_dir.is_dir():
            poluente = poluente_dir.name
    
            arquivos = [arq.stem for arq in poluente_dir.glob("*")]
    
            for id_mma in df_ids["ID_MMA"]:
                if any(str(arq).startswith(id_mma) for arq in arquivos):
                    id_to_poluentes.setdefault(id_mma, []).append(poluente)
    
    df_ids["POLUENTE"] = df_ids["ID_MMA"].map(id_to_poluentes).fillna("").apply(lambda x: ",".join(x) if isinstance(x, list) else "")
    
    return(df_ids)

In [4]:
df = pd.read_csv(os.getcwd()+'/data/DADOS_BRUTOS/PR/2017/FOZ2017/2017.xls', sep='\t', engine='python', encoding='latin1', header=2 )

df

,Data/Hora,CHUVA(mm),DV(º),PRESS(hPa),RADG(W/m²),TEMP(°C),TEMP INT(°C),UMID(%),VV(m/s),CH4(ppm),...,HCNM(ppm),HCT(ppm),MP10(µg/m³),NO(ppb),NO2(ppb),NOX(ppb),O3(ppb),PTS(µg/m³),SO2(ppb),Unnamed: 20
0,21/06/2017 00:00,insufic,"47,81",insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
1,21/06/2017 01:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
2,21/06/2017 02:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
3,21/06/2017 03:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
4,21/06/2017 04:00,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,...,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,insufic,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4652,31/12/2017 20:00,"0,00","91,25","977,88",insufic,"30,06","30,13","68,28","0,42",insufic,...,insufic,insufic,"54,00","20,56","14,74","35,31","8,72","76,00",insufic,NaN
4653,31/12/2017 21:00,"0,00","50,02","977,78",insufic,"29,45","30,15","73,63","0,47",insufic,...,insufic,insufic,"62,00","0,82","6,98","7,80","10,42","74,00",insufic,NaN
4654,31/12/2017 22:00,"0,00","356,79","978,03",insufic,"28,75","30,16","77,48","0,51",insufic,...,insufic,insufic,"17,00","0,23","6,52","6,75","11,36","13,00",insufic,NaN
4655,31/12/2017 23:00,"0,00","13,49","978,11",insufic,"28,27","30,30","78,45","0,31",insufic,...,insufic,insufic,"41,00","2,15","8,29","10,44","8,64","43,00",insufic,NaN


In [69]:
def ler_dados_parana_2024(dicionario,ano):
    
    #caminho =  os.getcwd()+'/data/DADOS_BRUTOS/PR/2024/'

    caminho = os.getcwd()+'/data/DADOS_BRUTOS/PR/'+ano+'/'
    
    arquivos = os.listdir(caminho)
    
    for arquivo in arquivos:
    
        if arquivo.endswith(('.xls', '.xlsx')):
            caminho_arquivo = os.path.join(caminho, arquivo)
            try:
                df = pd.read_excel(caminho_arquivo, header=3)
            except Exception as e:
                print(f"Tentando ler como texto: {arquivo}")
                df = pd.read_csv(caminho_arquivo, sep='\t', engine='python', encoding='latin1', header=2, on_bad_lines='skip')
                
            col_data = next(c for c in df.columns if c.startswith('Data'))
    
            df = (
                df.replace('-', np.nan)
                  .assign(**{c: pd.to_numeric(df[c], errors='coerce') for c in df.columns if c != col_data})
                  .groupby(col_data, as_index=False)
                  .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
            )

            if '5MIN' in arquivo:
                estacao = arquivo.split('_')[0]
            else:
                estacao = arquivo.split('2')[0]

            if df.columns[0] == 'Data/Hora':

                print(estacao)

                df = df[pd.to_datetime(df[col_data], errors='coerce').notna()]
    
                dicionario[ano][estacao] = df

            print(len(df))

    return dicionario

def adicionar_colunas_unidade(df):
    unidades = df.iloc[0]
    
    df = df.iloc[1:].reset_index(drop=True)
    
    for col, unidade in unidades.items():
        if pd.notna(unidade):
            df[f"{col}_UNIDADE"] = unidade
    
    return df

def num_para_hora(valor):
    try:
        h = int(valor)
        m = "30" if valor % 1 == 0.5 else "00"
        return f"{h}:{m}"
    except:
        return None
        
    return df

def ler_dados_parana_1998_2002(dicionario,ano):
    
    caminho = os.getcwd()+'/data/DADOS_BRUTOS/PR/'+ano+'/'
        
    arquivos = os.listdir(caminho)
    
    for arquivo in arquivos:

        print(arquivo)
    
        if any(Path(caminho+arquivo).iterdir()):
            pasta = os.listdir(caminho+arquivo)[0]
    
            df = pd.read_excel(caminho+arquivo+'/'+pasta,header=1)

            df = adicionar_colunas_unidade(df)

            for hora in ['H', 'HORA', 'Hora']:
                if hora in df.columns:
                    df[hora] = df[hora].astype(float).apply(num_para_hora)
                    break

            estacao = arquivo[:-4]
    
            dicionario[ano][estacao] = df 

            print(len(df))
    
        else:
            print('Não há nada em '+ caminho+arquivo)

    return dicionario

def ler_dados_mes_a_mes(caminho):

    tipos_arquivos_ignorar = ['.zip','.rar','.xls','.xlsx','.7z','testes','2016','.ipynb_checkpoints']

    df = pd.DataFrame()

    #print(caminho)
    #print(sorted(os.listdir(caminho)))

    estacao = caminho.split('/')[-2].split('2')[0]
    
    for arquivo in sorted(os.listdir(caminho)):
        df_mes = pd.DataFrame()
    
        if not any(p in arquivo for p in tipos_arquivos_ignorar) or any(p in arquivo for p in ['txt']):

            if '.txt' in arquivo:
                df_mes = pd.read_csv(caminho+arquivo, sep='\t', engine='python', encoding='latin1')
                #print(df_mes.head())
                #print(arquivo)
                df = pd.concat([df, df_mes], ignore_index=True)
            
            else:
                mes = arquivo[:2]
                base_path = os.path.join(caminho, arquivo)
                
                nomes_possiveis = [
                    [f"{estacao}1H_{mes}_{ano}.txt",0],
                    [f"{estacao}1H.txt",0],
                    [f"{estacao}1H_{mes}_{ano}.xls",3],
                    [f"{estacao}_1H.xls",2]
                ]

                for nome in nomes_possiveis:
                    full_path = os.path.join(base_path, nome[0])
                    
                    try:
                        df_mes = pd.read_csv(full_path, sep='\t', engine='python', encoding='latin1',header=nome[1])
                        break
                    except Exception:
                        try:
                            df_mes = pd.read_excel(full_path, engine='xlrd',header=nome[1])
                            break
                        except Exception:
                            continue

            if len(df_mes) == 0:

                print(caminho)
                #print(sorted(os.listdir(caminho)))
                #print(arquivo)
                print(df_mes.head())
                print('')
                
            df = pd.concat([df, df_mes], ignore_index=True)

            #df = pd.concat([df, df_mes], ignore_index=True)
           
    
    print('')
            
    
    return df

def verifica_numero(num):
    try:
        if num != np.nan:
            float(num)
            return True
    except (ValueError, TypeError):
        return False

def verifica_data(data):
    try:
        pd.to_datetime(data)
        return True
    except (ValueError, TypeError):
        return False

def ler_dados_parana_2003_2019(dicionario,ano):

    pastas_ignorar = ['IQA diário','IQA_IAP','2016','ARAUCARIA2018','ARAUCARIA2019','Thumbs.db','~$Validação_Maio_2014.xlsm','SIX1H_2017.zip','.ipynb_checkpoints']

    caminho = os.getcwd()+'/data/DADOS_BRUTOS/PR/'+ano+'/'
        
    arquivos = os.listdir(caminho)

    print('')
    print(ano)
    
    for arquivo in arquivos:

        if arquivo not in pastas_ignorar:

            print(arquivo)
            
            if any(nome.endswith(('.xls', '.xlsx')) for nome in os.listdir(caminho+arquivo)) and len(os.listdir(caminho+arquivo)) <= 3:
                print(os.listdir(caminho+arquivo))

                xlsx = [f for f in os.listdir(caminho+arquivo) if f.endswith('.xlsx')]
                xls = [f for f in os.listdir(caminho+arquivo) if f.endswith('.xls')]
                
                if xlsx:
                    estacao = xlsx[0] 
                elif xls:
                    estacao = xls[0]

                try:
                    if ano == '2003':
                        df = pd.read_excel(caminho+arquivo+'/'+estacao,header=1)
                    elif estacao == 'CIC2019.xlsx':
                        df = pd.read_excel(caminho+arquivo+'/'+estacao,header=2)
                    else:
                        df = pd.read_excel(caminho+arquivo+'/'+estacao)
                except Exception as e:
                    print(f"Tentando ler como texto: {arquivo}")
                    df = pd.read_csv(caminho+arquivo+'/'+estacao, sep='\t', engine='python', encoding='latin1', header=2)
                
                if not (verifica_numero(df[df.columns[0]].iloc[0]) or verifica_data(df[df.columns[0]].iloc[0])) or arquivo == 'SIX2017':

                    df = adicionar_colunas_unidade(df)

                print(len(df))
                    
                dicionario[ano][arquivo[:-4]] = df 

            elif 'IAP' not in arquivo:

                try:
                        
                    df = ler_dados_mes_a_mes(caminho+arquivo+'/')
                    
                    if not (verifica_numero(df[df.columns[0]].iloc[0]) or verifica_data(df[df.columns[0]].iloc[0])):
    
                        df = adicionar_colunas_unidade(df)
                
                    dicionario[ano][arquivo[:-4]] = df 

                    print(len(df))

                except Exception as e:
                    print(f"A seguinte pasta não existe: {caminho}{arquivo}")
                    
    return(dicionario)


In [65]:
len(pd.DataFrame())

0

In [240]:
estacoes_por_ano = {
    '1998': {},
    '1999': {},
    '2000': {},
    '2001': {},
    '2002': {},
    '2003': {},
    '2004': {},
    '2005': {},
    '2006': {},
    '2007': {},
    '2008': {},
    '2009': {},
    '2010': {},
    '2011': {},
    '2012': {},
    '2013': {},
    '2014': {},
    '2015': {},
    '2016': {},
    '2017': {},
    '2018': {},
    '2019': {},
    '2020': {},
    '2021': {},
    '2022': {},
    '2023': {},
    '2024': {}
}

for ano in ['1998','1999','2000',
            '2001','2002','2003','2004','2005','2006','2007','2008','2009','2010',
            '2011','2012','2013','2014','2015','2016','2017','2018','2019','2024']:

    if ano in ['1998','1999','2000','2001','2002']:
        estacoes_por_ano = ler_dados_parana_1998_2002(estacoes_por_ano,ano)
    elif ano in ['2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019']:
        estacoes_por_ano = ler_dados_parana_2003_2019(estacoes_por_ano,ano)
    elif ano in ['2024']:
        estacoes_por_ano = ler_dados_parana_2024(estacoes_por_ano,ano)

STC1998
2969
CIC1998
12067
STC1999
17472
CIC1999
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/1999/CIC1999
CIC2000
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2000/CIC2000
STC2000
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2000/STC2000
ASS2000
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2000/ASS2000
ASS2001
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2001/ASS2001
STC2001
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2001/STC2001
CIC2001
17520
CIC2002
8736
BOQ2002
Não há nada em /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2002/BOQ2002
ASS2002
8760

2003
CIC2003
['CIC1h-2003.xls']
8760
IAP2003
CSN2003

8771
UEG2003

6608
PAR2003

8771
RPR2003

3676
BOQ2003

8771

2004
UEG2004

8795
CIC2004

5142
RPR2004

8795
PAR2004

8795
BOQ2004

8795
CSN2004

8795
ASS2004

5142
IAP2004
STC2004

5887

2005
PAR2005

8771

/tmp/ipykernel_175285/3697526426.py:173: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  pd.to_datetime(data)


8760
PGA2017

8129
PAR2017
['PAR1H_2017.xls']
8760
CSN2017
['CSN1H_2017.xls']
8760
SIX2017
['SIX1H_2017.xls']
5832
ASS2017

A seguinte pasta não existe: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2017/ASS2017
UEG

A seguinte pasta não existe: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2017/UEG

2018
PGA2018

8841
LON2018

17517
CIC2018
['CIC2018.xls']
Tentando ler como texto: CIC2018
8761
ASS2018
['.ipynb_checkpoints', 'ASS18.xls']
Tentando ler como texto: ASS2018
8761
CVEL2018
/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2018/CVEL2018/
Empty DataFrame
Columns: [Report, Date/Time, 1:Temp, 2:O3, 3:CO, 4:NO, 5:NO2, 6:NOx, 7:SO2, 8:CH4 , 9:NMHC, 10:THC, 11:PM10, 12:PTS, 13:AT , 14:RH , 15:BP, 16:SR , 17:WS, 18:WD, 19:RAIN]
Index: []

[0 rows x 21 columns]


14295
CSN2018
['CSN2018.xls']
Tentando ler como texto: CSN2018


/tmp/ipykernel_175285/3697526426.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_mes], ignore_index=True)
/tmp/ipykernel_175285/3697526426.py:153: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_mes], ignore_index=True)


8761
UEG2018
['UEG2018.xls']
Tentando ler como texto: UEG2018
898
FOZ2018

15177
MRGA2018

17519
SIX2018
['.ipynb_checkpoints', 'SIX2018.xls']
Tentando ler como texto: SIX2018
8737

2019
CSN2019
['CSN2019.xls']
Tentando ler como texto: CSN2019
8761
CVEL2019

6783
RPR2019
['RPR2019.xls']
Tentando ler como texto: RPR2019
8761
FOZ2019
/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PR/2019/FOZ2019/
Empty DataFrame
Columns: [Report, Date/Time, 1:Temp, 2:O3, 3:CO, 4:NO, 5:NO2, 6:NOx, 7:SO2, 8:CH4 , 9:NMHC, 10:THC, 11:PM10, 12:PTS, 13:AT , 14:RH , 15:BP, 16:SR , 17:WS, 18:WD, 19:RAIN]
Index: []

[0 rows x 21 columns]


9057
LON2019


...

ECV


/tmp/ipykernel_175285/3697526426.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = df[pd.to_datetime(df[col_data], errors='coerce').notna()]


26485
Tentando ler como texto: MRGA_5MIN_2024.xls
MRGA
17280
Tentando ler como texto: MGA_5MIN_2024.xls
MGA
10368
Tentando ler como texto: RPR2024.xls
366
Tentando ler como texto: JDA_5MIN_2024.xls
JDA
10368
Tentando ler como texto: CLB_5MIN_2024.xls
CLB
10370
Tentando ler como texto: CVEL_5MIN_2024.xls
CVEL
10368
Tentando ler como texto: LDA_5MIN_2024.xls
LDA
10368
Tentando ler como texto: PARP2024.xls
366
Tentando ler como texto: PGA2024.xls
366
Tentando ler como texto: FNB_5MIN_2024.xls
FNB
10387
Tentando ler como texto: LON_5MIN_2024.xls
LON
10368
Tentando ler como texto: CIC2024.xls
CIC
3456
Tentando ler como texto: DCA_5MIN_2024.xls
DCA


/tmp/ipykernel_175285/3697526426.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = df[pd.to_datetime(df[col_data], errors='coerce').notna()]


26485
Tentando ler como texto: FOZ_5MIN_2024.xls
FOZ
10371
Tentando ler como texto: FOSPAR_5MIN_2024.xls
FOSPAR
10368
Tentando ler como texto: SIX2024.xls
366
Tentando ler como texto: CLB2024.xls
79


In [262]:
lista = []

def parse_datetime(x):
    for fmt in ('%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M', '%m/%d/%Y %I:%M:%S %p'):
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue
    return pd.to_datetime(x, errors='coerce')
                
def criar_datetime(df,tipo):

    if tipo == 'D':

        df['A'] = pd.to_numeric(df['A'], errors='coerce')

        if df['A'].iloc[0] < 2000:

            df['A'] = df['A'] + 2000
                
        df['H'] = df['H'].astype(str).str.split(':').str[0]
        
        df[['A', 'M', 'D', 'H']] = df[['A', 'M', 'D', 'H']].apply(pd.to_numeric, errors='coerce')
        
        df['datetime'] = pd.to_datetime(
            dict(year=df['A'], month=df['M'], day=df['D'], hour=df['H'].clip(upper=23)),
            errors='coerce'
        )
        
        df.loc[df['H'] == 24, 'datetime'] = df.loc[df['H'] == 24, 'datetime'] + pd.Timedelta(hours=1)
    
    elif tipo == 'DATA':

        df['datetime'] = pd.to_datetime(df['DATA']) + pd.to_timedelta(df['HORA'] + ':00')

        df = df.drop(columns=[c for c in ['ANO', 'MES', 'DIA', 'HORA'] if c in df.columns])
    
    elif tipo == 'Data':

        try:
            df['Data'] = pd.to_datetime(df['Data']).dt.date
        except:
            print(1)

        print(df.loc[df['Data'].astype(str).str.contains('--', na=False)])

        df['datetime'] = pd.to_datetime(df['Data']) + pd.to_timedelta(df['Hora'] + ':00')

    elif tipo == 'Data/Hora':
        
        df['Data/Hora'] = df['Data/Hora'].apply(parse_datetime)
        
        df['Data/Hora'] = df['Data/Hora'].dt.strftime('%Y-%m-%d %H:%M:%S')

        df['datetime'] = pd.to_datetime(df['Data/Hora'])

    elif tipo == 'Date/Time':
        
        df['Date/Time'] = df['Date/Time'].apply(parse_datetime)
        
        df['Date/Time'] = df['Date/Time'].dt.strftime('%Y-%m-%d %H:%M:%S')

        df['datetime'] = pd.to_datetime(df['Date/Time'])
        
    df = df.set_index("datetime")

    df = df.sort_index()
    
    df.insert(0, 'DATETIME', df.index)
    df.insert(1, 'ANO', df.index.year)
    df.insert(2, 'MES', df.index.month)
    df.insert(3, 'DIA', df.index.day)
    df.insert(4, 'HORA', df.index.hour)

    return df

lista_colunas = []

for ano in estacoes_por_ano.keys():
    
    print(ano)

    for estacao in estacoes_por_ano[ano].keys():

        #print(estacao)
        
        lista.append(estacao)

        lista_colunas = lista_colunas + list(estacoes_por_ano[ano][estacao].columns)
        
        if any(item in ['D','DATA','Data','Date/Time','Data/Hora'] for item in estacoes_por_ano[ano][estacao].columns):

            if 'D' in estacoes_por_ano[ano][estacao].columns:
                print('D')
                #estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'D')
                #print(estacoes_por_ano[ano][estacao])
            elif 'DATA' in estacoes_por_ano[ano][estacao].columns:
                print('DATA')
                #estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'DATA')
                #print(estacoes_por_ano[ano][estacao])
            elif 'Data' in estacoes_por_ano[ano][estacao].columns:
                print('Data')
                #estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'Data')
                #print(estacoes_por_ano[ano][estacao])
            elif 'Date/Time' in estacoes_por_ano[ano][estacao].columns:
                print('Date/Time')
                #estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'Date/Time')
                #print(estacoes_por_ano[ano][estacao])
            elif 'Data/Hora' in estacoes_por_ano[ano][estacao].columns:
                print('Data/Hora')
                #estacoes_por_ano[ano][estacao] = criar_datetime(estacoes_por_ano[ano][estacao], 'Data/Hora')
                #print(estacoes_por_ano[ano][estacao])
            else:
                print('ERRO')
    print('')

'''
lista1 = sorted(set(lista))

lista_pr = pd.read_csv(os.getcwd() + '/data/DADOS_ESTACOES/PR_estacoes.csv')

lista2 = sorted(lista_pr['ID_OEMA'])

iguais = sorted(set(lista1) & set(lista2))

so_lista1 = [x for x in lista1 if x not in iguais]
so_lista2 = [x for x in lista2 if x not in iguais]

col1 = iguais + so_lista1 + [np.nan] * len(so_lista2)
col2 = iguais + [np.nan] * len(so_lista1) + so_lista2

df = pd.DataFrame({'PR_dados': col1, 'PR_estacao': col2})

df'''

print(lista_colunas)
print(len(list(set(lista_colunas))))

1998
Data
Data

1999
Data

2000

2001
Data

2002
DATA
D

2003
D
D
D
D
D
D

2004
D
D
D
D
D
D
D
D

2005
D
D
D
D
D
D
D
D

2006
D
D
D
D
D
D
D
D

2007
D
D
D
D
D
D
D

2008
D
D
D
D
D
D

2009
D
D
D
D
D
D
D

2010
D
D
D
D
D
D
D
D

2011
D
D
D
D
D
D
D
D
D

2012
D
D
D
D
D
D
D
D
D

2013
D
D
D
D
D
D
D
D
D

2014
D
D
D
D
D
D
D
D
D

2015
D
D
D
D
D
D
D
D
D

2016
Date/Time
D
D
Date/Time
D
D
Date/Time
Data/Hora
D
D
D
D
D

2017
Date/Time
D
Data/Hora
Date/Time
D
Date/Time
D
D
D

2018
Date/Time
Date/Time
Data/Hora
Data/Hora
Date/Time
Data/Hora
Data/Hora
Date/Time
Date/Time
Data/Hora

2019
Data/Hora
Date/Time
Data/Hora
Date/Time
Date/Time
Date/Time
Date/Time
Data/Hora
Data/Hora
Data/Hora

2020

2021

2022

2023

2024
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora
Data/Hora

['Data', 'Hora', 'SO2', 'NO', 'NO2', 'Nox', 'O3', 'UVB', 'Temperatura', 'Umidade', 'Rad. Glob.', ' UVA', 'Press', 'V V', 'D V', 'SO2_UNID

In [30]:
estacoes_dicionario.keys()

for chave in estacoes_dicionario.keys():

    print(chave)
    print(len(estacoes_dicionario[chave]))
    print(estacoes_dicionario[chave].columns[0])
    print(estacoes_dicionario[chave].iloc[0, 0])
    print(estacoes_dicionario[chave].iloc[-1, 0])
    print("")

    estacoes_por_ano

,D,M,A,H,SO2,NO,NO2,O3,CO,PTS,...,THC,CH4,NMHC,TEMP,UMID,PRESS,VV,DV,RADG,CHUVA
0,NaN,NaN,NaN,NaN,ppb,ppb,ppb,ppb,ppm,µg/m³,...,ppm,ppm,ppm,°C,%,mBar,m/s,graus,W/m2,mm
1,1.0,1.0,17.0,1.0,0,NaN,NaN,7.8,0.46,NaN,...,1.39,1.33,NaN,23.2,99.9,923.3,1.2,232.4,7.7,NaN
2,1.0,1.0,17.0,2.0,0.2,NaN,NaN,15,0.38,NaN,...,1.38,1.26,NaN,22,99.9,922.2,1.2,189.2,6,NaN
3,1.0,1.0,17.0,3.0,0.6,NaN,NaN,10.7,0.37,NaN,...,1.35,1.26,NaN,21.9,99.9,921.5,0.9,274.3,5.3,NaN
4,1.0,1.0,17.0,4.0,0.4,NaN,NaN,11.1,0.41,NaN,...,1.35,1.26,NaN,21.7,99.9,921.1,1.2,275,4,NaN


In [254]:
list(estacoes_por_ano[ano][estacao].columns)


['DATETIME',
 'ANO',
 'MES',
 'DIA',
 'HORA',
 'Data/Hora',
 'CHUVA(mm)',
 'DV(º)',
 'PRESS(hPa)',
 'RADG(W/m²)',
 'TEMP(°C)',
 'UMID(%)',
 'VV(m/s)',
 'CO(ppm)',
 'MP10(µg/m³)',
 'PM_2_5(µg/m³)',
 'SO2(ppb)',
 'TRS(ppb)',
 'Unnamed: 13']

In [2]:
def rep_temp(df,agrupamento,criterio,periodo_ref):

    resultados = []
    
    for chave, dados in df.groupby(agrupamento):
        qntd_valor = dados['VALOR'].notna().sum()  

        if criterio == 'HORA':
            qntd_tempo = 24
        elif criterio == 'DIA':
            if chave[1] in [4,6,9,11]:
                qntd_tempo = 30
            elif chave[1] in [1,3,5,7,8,10,12]:
                qntd_tempo = 31
            else:
                if (chave[0] % 4 == 0 and chave[0] % 100 != 0) or (chave[0] % 400 == 0):
                    qntd_tempo = 29
                else:
                    qntd_tempo = 28
        
        if qntd_valor >= (2/3) * qntd_tempo:
            if periodo_ref == "8horas":
                media = dados["VALOR"].rolling(window=8, min_periods=1).mean().max()
            else:
                media = periodo_ref(dados["VALOR"])
            rep = True
        else:
            media = np.nan   
            rep = False

        prcnt = 100*qntd_valor/qntd_tempo

        resultados.append((*chave, media, rep, prcnt))

    return resultados

def conta_dias_quadrimestre(ano,quadrimestre):

    if quadrimestre == 1:
        if (ano % 4 == 0 and ano % 100 != 0) or (ano % 400 == 0):
            dias = 121
        else:
            dias = 120
    elif quadrimestre == 2:
        dias = 123
    else:
        dias = 122

    return dias

def rep_temp_ano(df,agrupamento,criterio):

    resultados = []
    
    for chave, dados in df.groupby(agrupamento):
        qntd_valor = dados['VALOR'].notna().sum()
        qntd_tempo = conta_dias_quadrimestre(chave[0],chave[1])         
        
        if qntd_valor >= (1/2) * qntd_tempo:
            rep = True
        else:
            rep = False
    
        resultados.append((*chave, rep))

    return resultados
    
df_estacoes_rep_temporal = pd.DataFrame({
                    'ID_MMA_COMPLETO':[],
                    'PRCNT_REP_DIA':[],
                    'PRCNT_REP_MES':[],
                    'PRCNT_REP_ANO':[]
                })
                
for pol in df_rep_temporal.index:
    
    path = os.getcwd()+'/data/MQAr/' + pol + '/'

    print(pol)
    
    if os.path.isdir(path) and os.listdir(path):
        
        arquivos = os.listdir(path)
    
        for estacao in arquivos:

            if estacao.endswith('.csv'):

                print(estacao)

                df = pd.read_csv(path+estacao)

                df["VALOR"] = pd.to_numeric(df["VALOR"], errors="coerce")

                resultados_24 = rep_temp(df,['ANO','MES','DIA'],'HORA',np.mean)

                df_24 = pd.DataFrame(resultados_24, columns=['ANO', 'MES', 'DIA', 'VALOR', 'REP_DIA','PRCNT_REP'])

                df_24['DATETIME'] = pd.to_datetime(
                        dict(year=df_24["ANO"], month=df_24["MES"], day=df_24["DIA"])
                    )
            
                df_24 = df_24[['DATETIME','ANO','MES','DIA','VALOR','REP_DIA','PRCNT_REP']]

                if df_rep_temporal['DIA'][pol] == 'dia':
                    df_dia = df_24
                elif df_rep_temporal['DIA'][pol] == '8horas':
                    resultados_dia = rep_temp(df,['ANO','MES','DIA'],'HORA','8horas')

                    df_dia = pd.DataFrame(resultados_dia, columns=['ANO', 'MES', 'DIA', 'VALOR', 'REP_DIA','PRCNT_REP'])

                    df_dia['DATETIME'] = pd.to_datetime(dict(year=df_dia["ANO"], month=df_dia["MES"], day=df_dia["DIA"]))

                    df_dia = df_dia[['DATETIME','ANO','MES','DIA','VALOR','REP_DIA','PRCNT_REP']]
                        
                else:
                    resultados_dia = rep_temp(df,['ANO','MES','DIA'],'HORA',np.max)

                    df_dia = pd.DataFrame(resultados_dia, columns=['ANO', 'MES', 'DIA', 'VALOR', 'REP_DIA','PRCNT_REP'])

                    df_dia['DATETIME'] = pd.to_datetime(dict(year=df_dia["ANO"], month=df_dia["MES"], day=df_dia["DIA"]))

                    df_dia = df_dia[['DATETIME','ANO','MES','DIA','VALOR','REP_DIA','PRCNT_REP']]

                df_dia.to_csv(os.getcwd()+'/data/MQAr_averages/'+df_rep_temporal['DIA'][pol]+'/'+pol+'/'+estacao,index=False)

                resultados_mes = rep_temp(df_24,['ANO','MES'],'DIA',np.mean)

                df_mes = pd.DataFrame(resultados_mes, columns=['ANO', 'MES', 'VALOR', 'REP_MES','PRCNT_REP'])

                df_mes['DATETIME'] = pd.to_datetime(dict(year=df_mes["ANO"], month=df_mes["MES"], day=1))
            
                df_mes = df_mes[['DATETIME','ANO','MES','VALOR','REP_MES','PRCNT_REP']]

                df_mes.to_csv(os.getcwd()+'/data/MQAr_averages/'+df_rep_temporal['MES'][pol][:6]+'/'+pol+'/'+estacao,index=False)

                condicoes = [
                    (df_dia['MES'] <= 4),
                    (df_dia['MES'] >= 5) & (df_dia['MES'] <= 8),
                    (df_dia['MES'] >= 9)
                ]
                
                quadrimestre = [1, 2, 3]
                
                df_dia['QUADRIMESTRE'] = np.select(condicoes, quadrimestre)

                resultados_quad = rep_temp_ano(df_dia,['ANO','QUADRIMESTRE'],'QUADRIMESTRE')

                df_quad = pd.DataFrame(resultados_quad, columns=['ANO', 'QUADRIMESTRE', 'REP_QUAD'])
                
                df_ano_quad = df_quad.groupby("ANO", as_index=False).agg({"REP_QUAD": lambda x: x.sum() == 3})

                resultados = []
                
                for ano, dados in df_dia.groupby(['ANO']):
                    qntd_valor = dados['VALOR'].notna().sum()
                    if (ano[0] % 4 == 0 and ano[0] % 100 != 0) or (ano[0] % 400 == 0):
                        dias = 366
                    else:
                        dias = 365
                    prcnt_rep = (100*qntd_valor/dias)
                    if df_ano_quad.loc[df_ano_quad["ANO"] == ano[0], "REP_QUAD"].values[0] == True:
                        media = dados['VALOR'].mean()
                        rep = True
                    else:
                        media = np.nan
                        rep = False
                    resultados.append((*ano, media, rep, prcnt_rep))
                
                df_ano = pd.DataFrame(resultados, columns=['ANO', 'VALOR','REP_ANO','PRCNT_REP'])
                
                df_ano['DATETIME'] = pd.to_datetime(dict(year=df_ano["ANO"], month=1, day=1))
            
                df_ano = df_ano[['DATETIME','ANO','VALOR','REP_ANO','PRCNT_REP']]

                df_ano.to_csv(os.getcwd()+'/data/MQAr_averages/'+df_rep_temporal['ANO'][pol][:5]+'/'+pol+'/'+estacao, index=False)

                prcnt_dia = 100 * df_dia['VALOR'].notna().sum() / len(df_dia)
                prcnt_mes = 100 * df_mes['VALOR'].notna().sum() / len(df_mes)
                prcnt_ano = 100 * df_ano['VALOR'].notna().sum() / len(df_ano)

                prcnt_estacao = {'ID_MMA_COMPLETO': estacao[:-4], 'PRCNT_REP_DIA': prcnt_dia, 'PRCNT_REP_MES': prcnt_mes, 'PRCNT_REP_ANO': prcnt_ano}

                df_estacoes_rep_temporal = pd.concat([df_estacoes_rep_temporal, pd.DataFrame([prcnt_estacao])], ignore_index=True)

df_estacoes_rep_temporal.to_csv(os.getcwd()+'/data/MQAr_averages/REP_TEMPORAL.csv', index=False)
                
                

MP10
SP0248RA001.csv
RJ0031RA001.csv


KeyboardInterrupt: 

In [70]:
df_estacoes_rep_temporal

,ID_MMA_COMPLETO,PRCNT_REP_DIA,PRCNT_REP_MES,PRCNT_REP_ANO
0,SP0248RA001,96.243112,95.959596,88.235294
1,RJ0031RA001,88.713704,90.800000,80.952381
2,RJ0282RA001,84.054054,76.923077,0.000000


In [71]:
estacao

'SP0085RA001.csv'

In [74]:
condicoes = [
    (df_dia['MES'] <= 4),
    (df_dia['MES'] >= 5) & (df_dia['MES'] <= 8),
    (df_dia['MES'] >= 9)
]

quadrimestre = [1, 2, 3]

df_dia['QUADRIMESTRE'] = np.select(condicoes, quadrimestre)

resultados_quad = rep_temp_ano(df_dia,['ANO','QUADRIMESTRE'],'QUADRIMESTRE')

df_quad = pd.DataFrame(resultados_quad, columns=['ANO', 'QUADRIMESTRE', 'REP_QUAD'])

df_ano_quad = df_quad.groupby("ANO", as_index=False).agg({"REP_QUAD": lambda x: x.sum() == 3})
                

In [83]:
df_ano

,ANO,VALOR,REP_ANO
0,"(2009,)",NaN,False
1,"(2010,)",76.227778,True
2,"(2011,)",75.494915,True
3,"(2012,)",77.752161,True
4,"(2013,)",69.000000,True
5,"(2014,)",89.419890,True
6,"(2015,)",67.670360,True
7,"(2016,)",73.575385,True
8,"(2017,)",72.285714,True
9,"(2018,)",67.177465,True


In [41]:
df_pivot = df_mes.pivot(index='ANO', columns='MES', values='PRCNT_REP')

df_pivot = df_pivot.reindex(sorted(df_pivot.columns), axis=1)

df_pivot = df_pivot.reset_index()

df_pivot

MES,ANO,1,2,3,4,5,6,7,8,9,10,11,12
0,2009,NaN,NaN,NaN,53.333333,100.000000,100.000000,100.000000,87.096774,86.666667,100.000000,96.666667,100.000000
1,2010,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,96.666667,87.096774,100.000000,100.000000
2,2011,100.000000,100.000000,100.000000,100.000000,80.645161,0.000000,41.935484,100.000000,100.000000,96.774194,83.333333,67.741935
3,2012,58.064516,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,80.645161
4,2013,96.774194,96.428571,80.645161,100.000000,100.000000,93.333333,93.548387,100.000000,100.000000,93.548387,100.000000,100.000000
5,2014,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.322581,100.000000,100.000000,100.000000,100.000000
6,2015,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.000000,96.774194
7,2016,77.419355,100.000000,100.000000,100.000000,100.000000,86.666667,54.838710,100.000000,100.000000,100.000000,93.333333,54.838710
8,2017,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,96.774194,100.000000,100.000000
9,2018,100.000000,100.000000,100.000000,66.666667,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [9]:
df = pd.read_csv(os.getcwd()+'/data/MQAr/MP10/ES0001RA001.csv')

def create_QAQCMMA_VALOR(df,pol):

    df = df.rename(columns={'VALOR':'VALOR_ORIGINAL'})

    flags_invalidos = ['!', 'IF', 'IO', 'IC', 'I%', 'IL', 'IE', 'IS', 'IU', 'IM', 'IP', 'ID', 'IT', 'IR', 
                       'Fora da Faixa de Medição', 'Disabilitada Temporariamente', 'Inválido', 
                       'Insuficientes', 'Inexistente']

    df['QAQC_INTERNO'] = ~df['QAQC_INTERNO'].isin(flags_invalidos)
    
    DEFAULT_RANGE_LIMITS = {
        "O3": (0, 500),
        "CO": (0, 50),
        "NO2": (0, 1000),
        "NOX": (0, 2000),
        "SO2": (0, 1000),
        "MP25": (0, 1000),
        "MP10": (0, 2000),
    }

    df['QAQC_MMA'] = df['QAQC_INTERNO']

    if pol in list(DEFAULT_RANGE_LIMITS.keys()):
        lim_min = DEFAULT_RANGE_LIMITS[pol][0]
        lim_max = DEFAULT_RANGE_LIMITS[pol][1]
    else:
        lim_min = 0
        lim_max = np.inf

    df['VALOR'] = df['VALOR_ORIGINAL']

    df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce')
    
    df.loc[df['QAQC_MMA'] & (df['VALOR'].isna() | (df['VALOR'] < lim_min) | (df['VALOR'] > lim_max)), 'QAQC_MMA'] = False
    
    df.loc[~df['QAQC_MMA'], 'VALOR'] = np.nan
    
    df = df[['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR', 'VALOR_ORIGINAL', 'UNIDADE', 'QAQC_INTERNO', 'QAQC_MMA']]

    return df

    
    

In [15]:
DEFAULT_RANGE_LIMITS = {
        "O3": (0, 500),
        "CO": (0, 50),
        "NO2": (0, 1000),
        "NOX": (0, 2000),
        "SO2": (0, 1000),
        "MP25": (0, 1000),
        "MP10": (0, 2000),
    }

DEFAULT_RANGE_LIMITS['MP10'][1]

2000

In [17]:
list(DEFAULT_RANGE_LIMITS.keys())

['O3', 'CO', 'NO2', 'NOX', 'SO2', 'MP25', 'MP10']

In [29]:
pol = 'CCCC'

df['QAQC_MMA'] = df['QAQC_INTERNO']

if pol in list(DEFAULT_RANGE_LIMITS.keys()):
    lim_min = DEFAULT_RANGE_LIMITS[pol][0]
    lim_max = DEFAULT_RANGE_LIMITS[pol][1]
else:
    lim_min = 0
    lim_max = np.inf

In [30]:
df['VALOR'] = df['VALOR_ORIGINAL']

df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce')

df.loc[df['QAQC_MMA'] & (df['VALOR'].isna() | (df['VALOR'] < lim_min) | (df['VALOR'] > lim_max)), 'QAQC_MMA'] = False

df.loc[~df['QAQC_MMA'], 'VALOR'] = np.nan

df = df[['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR', 'VALOR_ORIGINAL', 'UNIDADE', 'QAQC_INTERNO', 'QAQC_MMA']]


In [32]:
df.columns

Index(['DATETIME', 'ANO', 'MES', 'DIA', 'HORA', 'VALOR_ORIGINAL', 'UNIDADE',
       'QAQC_INTERNO', 'QAQC_MMA', 'VALOR'],
      dtype='object')

In [36]:
import pandas as pd
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

def ppb_to_ug(df,pol):

    if pol == 'so2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 2661260.49/10**6        

    elif pol == 'no2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1911038.92/10**6        

    elif pol == 'o3':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1993889.17/10**6        
    
    df.loc[:, "Unidade"] = "ug/m3"

    return df

def rectify_MT(path):

    dict_pols_stat = defaultdict(list)

    files = os.listdir(path)
    
    print(files)
    
    for item in files:
        
        estacao = " ".join(item.split('-')[1].split('.')[0].split('_')[0:2])
    
        print(estacao)
    
        df = pd.read_excel(path+item)
    
        df = df.drop(columns=['Nome da estação'])
    
        lista_pols = set(df['Poluente'])
    
        for pol in lista_pols:
    
            df_pol = df[df["Poluente"] == pol]
            
            if pol in ['no2','so2','o3']:
    
                df_pol = ppb_to_ug(df_pol,pol)
    
            df_pol_hora = df_pol.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"])
            
            df_pol_hora = df_pol_hora.filter(lambda g: len(g) >= 9)
            
            df_pol = (
                df_pol_hora.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"], as_index=False)
                      .agg({"Valor": "mean"})
            )
    
            df_pol['QAQC_INTERNO'] = None
    
            df_pol = df_pol.rename(columns={'Ano':'ANO',
                                            'Mes':'MES',
                                            'Dia':'DIA',
                                            'Hora':'HORA',
                                            'Unidade':'UNIDADE',
                                            'Valor':'VALOR'})
    
            for col in ["ANO", "MES", "DIA", "HORA"]:
                df_pol[col] = pd.to_numeric(df_pol[col], errors="coerce").astype("Int64")
            
            dict_pols_stat[estacao+'_'+pol].append(df_pol)

    dict_pols_MT = {
        'co':'CO',
        'no2': 'NO2',
        'so2': 'SO2',
        'o3': 'O3',
        'pm2p5':'MP25',
        'pm10': 'MP10'
    }

    dict_formatado = {}
    
    for chave in dict_pols_stat.keys():
        
        lista_dfs = dict_pols_stat[chave]
        
        df = pd.concat(lista_dfs, ignore_index=True)
    
        df["DATETIME"] = pd.to_datetime(
            df.apply(lambda r: f"{r.ANO}-{r.MES}-{r.DIA} {r.HORA}:00:00", axis=1)
        )
        df = df.set_index("DATETIME")
    
        df = df.sort_index()
        
        lista_horas = pd.date_range(
            start=df.index.min(), 
            end=df.index.max(), 
            freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
        
        if len(lista_horas) != len(df):
            df = df.reindex(pd.DatetimeIndex(lista_horas))
    
        df['DATETIME'] = df.index
    
        df = df[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']]
        
        dict_formatado[chave] = df
        
        primeiros_valores = {}
    
    for chave, df in dict_formatado.items():
        
        if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
            
            linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
            primeiros_valores[chave] = linha_valida["DATETIME"]
    
    codigo_estacao_MT = {}
    
    for chave in primeiros_valores.keys():
        
        station = chave.split('_')[0]
        data = primeiros_valores[chave]
        
        if station in codigo_estacao_MT:
            if data <= codigo_estacao_MT[station]:
                codigo_estacao_MT[station] = data
        else:
            codigo_estacao_MT[station] = data
    
    sorted_items = sorted(
        codigo_estacao_MT.items(),
        key=lambda x: (x[1], x[0])
    )
    
    codigo_estacao_MT = {}
    for i, (nome, ts) in enumerate(sorted_items, start=1):
        codigo = f"MT{i:04d}"
        codigo_estacao_MT[nome] = codigo
        
    for chave, df in dict_formatado.items():
        
        estacao = codigo_estacao_MT[chave.split('_')[0]]
        
        cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == dict_pols_MT[chave.split('_')[-1]], 'COD_POLUENTE'].values[0]
        
        nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]

        df = create_QAQCMMA_VALOR(df,nome_pasta)
        
        df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'IA'+str(cod_pol).zfill(3)+'.csv',index=False)

    dict_stations_MT = {
        'Sema':'CPA - SEMA - CBA',
        'BEA CBA': 'Dom Aquino - BEA - CBA',
        'CBM VG': 'Água Limpa - CBM - VG',
        'Mae Bonifacia': 'Duque de Caxias - Pq Mãe Bonifácia - CBA',
        'UFMT':'Boa Esperança - UFMT - CBA'
    }
    
    df_ids = pd.DataFrame({
        'ID_OEMA': codigo_estacao_MT.keys(),
        'ID_MMA':list(codigo_estacao_MT.values())})
    
    df_ids["ID_OEMA"] = df_ids["ID_OEMA"].replace(dict_stations_MT)

    print(df_ids)
    
    return df_ids

funcoes = {
    'MT': rectify_MT
}

#lista_estados = ['SC','RS','MT','DF','MA','BA','ES','MG','SP','RJ']
lista_estados = ['MT']

tabela_ids = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv')
tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')

for estado in lista_estados:
    
    path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

    df_ids = funcoes[estado](path)
    
    create_df_estacao(estado,df_ids)

['dados_monitoramento-Sema.xlsx', 'dados_monitoramento-BEA_CBA_24-25.xlsx', 'dados_monitoramento-BEA_CBA_22-23.xlsx', 'dados_monitoramento-CBM_VG_22-23.xlsx', 'dados_monitoramento-CBM_VG_24-25.xlsx', 'dados_monitoramento-Mae_Bonifacia_22-23.xlsx', 'dados_monitoramento-Mae_Bonifacia_24-25.xlsx', 'dados_monitoramento-UFMT.xlsx']
Sema
BEA CBA


KeyboardInterrupt: 

In [38]:
df = df_dia.groupby(['ANO'])

NameError: name 'df_dia' is not defined

In [72]:

tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')

In [73]:
for pol in tabela_pols['NOME_PASTA'].unique():
    print(pol)

MP10
MP25
SO2
NO2
O3
FMC
CO
PTS
CHUMBO
BENZENO
CH4
ERT
ETILBENZENO
H2S
NH3
HCNM
NO
NOX
MP1 
ACETAL
FORMAL
TOLUENO
XILENO
VOC
OXILENO
HCT
MPXILENO


In [2]:
import pandas as pd
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [18]:
funcoes = {
    
    'MT': rectify_MT,
}

lista_estados = ['MT']

tabela_ids = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv')
tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')


In [72]:

estado = 'DF'
  
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

#df_ids = funcoes[estado](path)
    
#create_df_estacao(estado,df_ids)

In [52]:
dict_pols_stat = defaultdict(list)

files = os.listdir(path)

print(files)

for item in files:
    
    estacao = " ".join(item.split('-')[1].split('.')[0].split('_')[0:2])

    print(estacao)

    df = pd.read_excel(path+item)

    df = df.drop(columns=['Nome da estação'])

    lista_pols = set(df['Poluente'])

    for pol in lista_pols:

        df_pol = df[df["Poluente"] == pol]
        
        if pol in ['no2','so2','o3']:

            df_pol = ppb_to_ug(df_pol,pol)

        df_pol_hora = df_pol.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"])
        
        df_pol_hora = df_pol_hora.filter(lambda g: len(g) >= 9)
        
        df_pol = (
            df_pol_hora.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"], as_index=False)
                  .agg({"Valor": "mean"})
        )

        df_pol['QAQC_INTERNO'] = None

        df_pol = df_pol.rename(columns={'Ano':'ANO',
                                        'Mes':'MES',
                                        'Dia':'DIA',
                                        'Hora':'HORA',
                                        'Unidade':'UNIDADE',
                                        'Valor':'VALOR'})

        for col in ["ANO", "MES", "DIA", "HORA"]:
            df_pol[col] = pd.to_numeric(df_pol[col], errors="coerce").astype("Int64")
        
        dict_pols_stat[estacao+'_'+pol].append(df_pol)



['dados_monitoramento-Sema.xlsx', 'dados_monitoramento-BEA_CBA_24-25.xlsx', 'dados_monitoramento-BEA_CBA_22-23.xlsx', 'dados_monitoramento-CBM_VG_22-23.xlsx', 'dados_monitoramento-CBM_VG_24-25.xlsx', 'dados_monitoramento-Mae_Bonifacia_22-23.xlsx', 'dados_monitoramento-Mae_Bonifacia_24-25.xlsx', 'dados_monitoramento-UFMT.xlsx']
Sema
BEA CBA
BEA CBA
CBM VG
CBM VG
Mae Bonifacia
Mae Bonifacia
UFMT


In [46]:
def ppb_to_ug(df,pol):

    if pol == 'so2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 2661260.49/10**6        

    elif pol == 'no2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1911038.92/10**6        

    elif pol == 'o3':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1993889.17/10**6        
    
    df.loc[:, "Unidade"] = "ug/m3"

    return df

In [60]:
dict_pols_MT = {
    'co':'CO',
    'no2': 'NO2',
    'so2': 'SO2',
    'o3': 'O3',
    'pm2p5':'MP25',
    'pm10': 'MP10'
}

dict_formatado = {}

for chave in dict_pols_stat.keys():
    
    lista_dfs = dict_pols_stat[chave]
    
    df = pd.concat(lista_dfs, ignore_index=True)

    df["DATETIME"] = pd.to_datetime(
        df.apply(lambda r: f"{r.ANO}-{r.MES}-{r.DIA} {r.HORA}:00:00", axis=1)
    )
    df = df.set_index("DATETIME")

    df = df.sort_index()
    
    lista_horas = pd.date_range(
        start=df.index.min(), 
        end=df.index.max(), 
        freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
    
    if len(lista_horas) != len(df):
        df = df.reindex(pd.DatetimeIndex(lista_horas))

    df['DATETIME'] = df.index

    df = df[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']]
    
    dict_formatado[chave] = df

primeiros_valores = {}

for chave, df in dict_formatado.items():
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_MT = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_MT:
        if data <= codigo_estacao_MT[station]:
            codigo_estacao_MT[station] = data
    else:
        codigo_estacao_MT[station] = data

sorted_items = sorted(
    codigo_estacao_MT.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_MT = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"MT{i:04d}"
    codigo_estacao_MT[nome] = codigo
    
for chave, df in dict_formatado.items():
    
    estacao = codigo_estacao_MT[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == dict_pols_MT[chave.split('_')[-1]], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})

return df_ids

/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = p

SyntaxError: 'return' outside function (2479180702.py, line 85)

In [61]:
df_ids

,ID_OEMA,ID_MMA
0,Mae Bonifacia,MT0001
1,BEA CBA,MT0002
2,CBM VG,MT0003
3,Sema,MT0004
4,UFMT,MT0005


In [185]:
from datetime import timedelta

def fix_24h(row):
    if isinstance(row, str) and row.startswith("24:"):
        # substitui 24: por 00:
        new_str = row.replace("24:", "00:", 1)
        # converte para datetime
        dt = pd.to_datetime(new_str, errors="coerce")
        # adiciona 1 dia
        if pd.notna(dt):
            dt += timedelta(days=1)
        return dt
    else:
        return pd.to_datetime(row, errors="coerce")



In [70]:
dict_stations_MT = {
        'Sema':'CPA - SEMA - CBA',
        'BEA CBA': 'Dom Aquino - BEA - CBA',
        'CBM VG': 'Água Limpa - CBM - VG',
        'Mae Bonifacia': 'Duque de Caxias - Pq Mãe Bonifácia - CBA',
        'UFMT':'Boa Esperança - UFMT - CBA'
    }
    
df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})


df_ids["ID_OEMA"] = df_ids["ID_OEMA"].replace(dict_stations_MT)


df_ids['ID_OEMA']

0    Duque de Caxias - Pq Mãe Bonifácia - CBA
1                      Dom Aquino - BEA - CBA
2                       Água Limpa - CBM - VG
3                            CPA - SEMA - CBA
4                  Boa Esperança - UFMT - CBA
Name: ID_OEMA, dtype: object

In [71]:
df_ids

,ID_OEMA,ID_MMA
0,Duque de Caxias - Pq Mãe Bonifácia - CBA,MT0001
1,Dom Aquino - BEA - CBA,MT0002
2,Água Limpa - CBM - VG,MT0003
3,CPA - SEMA - CBA,MT0004
4,Boa Esperança - UFMT - CBA,MT0005


In [120]:
estado = 'DF'

In [249]:
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

path = path + 'Monitor Report 2024_FINAL.xlsx'

df = pd.read_excel(path)

df.iloc[1] = df.iloc[1].ffill()

poluentes = ['CO_ppm','NO2_ug/m3','NO_ug/m3','NOx_ug/m3','O3_ug/m3','PM10','PM25','PTS','SO2_ug/m3']

dict_pols = {'CO_ppm':'CO',
             'NO2_ug/m3':'NO2',
             'NO_ug/m3':'NO',
             'NOx_ug/m3':'NOX',
             'O3_ug/m3':'O3',
             'PM10':'MP10',
             'PM25':'MP25',
             'PTS':'PTS',
             'SO2_ug/m3':'SO2'}

df.columns = df.iloc[1]

df = df.drop(index=[0, 1]).reset_index(drop=True)

df = df.rename(columns={'Date Time':'DATETIME'})

estacoes = set(df.columns[1:])

dict_pols_stat = defaultdict(list)

for estacao in estacoes:

    df_estacao = df[["DATETIME",estacao]]

    df_estacao.columns = [df_estacao.columns.tolist()[0]] + df_estacao.iloc[0, 1:].tolist()

    df_estacao = df_estacao.drop(index=[0]).reset_index(drop=True)

    for pol in poluentes:

        if pol in df_estacao.columns:
            
            df_pol = df_estacao[["DATETIME",pol]]

            df_pol['UNIDADE'] = df_pol[pol][0]

            df_pol = df_pol.drop(index=[0]).reset_index(drop=True)

            df_pol = df_pol[df_pol["DATETIME"].astype(str).str.contains(r"\d", na=False)].reset_index(drop=True)

            df_pol['DATETIME'] = df_pol['DATETIME'].apply(fix_24h)

            df_pol = df_pol.rename(columns={pol:'VALOR'})

            df_pol.index = df_pol['DATETIME']

            lista_horas = pd.date_range(
                start=df_pol.index.min(), 
                end=df_pol.index.max(), 
                freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
            
            if len(lista_horas) != len(df_pol):
                df_pol = df_pol.reindex(pd.DatetimeIndex(lista_horas))
            
            df_pol['QAQC_INTERNO'] = None
            
            df_pol.insert(1, 'ANO', df_pol.index.year)
            df_pol.insert(2, 'MES', df_pol.index.month)
            df_pol.insert(3, 'DIA', df_pol.index.day)
            df_pol.insert(4, 'HORA', df_pol.index.hour)

            pol = dict_pols[pol]

            df_pol['VALOR'] = pd.to_numeric(df_pol['VALOR'], errors='coerce')

            df_pol = df_pol[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']] 

            dict_pols_stat[estacao+'_'+pol] = df_pol
            

/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pol['UNIDADE'] = df_pol[pol][0]
/tmp/ipykernel_564959/1508039512.py:14: UserWarning: Parsing dates in %d:%M %m/%H/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(row, errors="coerce")
/tmp/ipykernel_564959/901552261.py:57: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

In [258]:
primeiros_valores = {}

for chave, df in dict_pols_stat.items():  
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_DF = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_DF:
        if data <= codigo_estacao_DF[station]:
            codigo_estacao_DF[station] = data
    else:
        codigo_estacao_DF[station] = data

sorted_items = sorted(
    codigo_estacao_DF.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_DF = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"DF{i:04d}"
    codigo_estacao_DF[nome] = codigo
    
for chave, df in dict_pols_stat.items():
    
    estacao = codigo_estacao_DF[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == chave.split('_')[-1], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

dict_oemas = {
    'Estação CRAS FERCAL': 'Fercal CRAS',
    'Estação Escola':	   'Fercal Escola'}

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_DF.keys(),
    'ID_MMA':list(codigo_estacao_DF.values())})

df_ids['ID_OEMA'] = df_ids['ID_OEMA'].replace(dict_oema)

return df_ids

In [275]:
df_ufs = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/IBGE_UFS_CODIGOS.csv')
    
cod_uf =  df_ufs.loc[df_ufs['UF'] == uf, 'CODIGOS'].values[0]

print(cod_uf)

if os.path.exists('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv'):

    df_estacao = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv')

    df_estacao['ID_MMA'] = df_estacao['ID_OEMA'].map(df_ids.set_index('ID_OEMA')['ID_MMA'])

else:

    colunas = ['ID_OEMA', 'UF', 'ID_MMA', 'COD_UF_IBGE', 'CIDADE', 'CD_MUN',
               'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE', 'LATITUDE',
               'LONGITUDE', 'MOBILIDADE', 'REALOCACAO', 'MARCA', 'CATEGORIA',
               'FUNCIONAMENTO', 'METODO', 'FINALIDADE', 'POLUENTE',
               'INICIO', 'STATUS', 'FIM', 'CALIBRACAO', 'OBS_CALIBRACAO', 'MONITORAR',
               'FONTE', 'OBS_GERAIS','DADOS_MONITORAMENTO','RECONHECIDA','REP_ESPACIAL_DECLARADA']
    
    df_estacao = pd.DataFrame(columns=colunas)

df_ids = pol_to_station(df_ids)

mapa = dict(zip(df_ids['ID_MMA'], df_ids['POLUENTE']))

df_estacao['POLUENTE'] = df_estacao['ID_MMA'].map(mapa).fillna(df_estacao['POLUENTE'])

#df_estacao = df_estacao.reindex(df_ids.index)

#df_estacao[["ID_MMA", "ID_OEMA", "POLUENTE"]] = df_ids[["ID_MMA", "ID_OEMA", "POLUENTE"]].values

df_estacao.loc[:, "COD_UF_IBGE"] = cod_uf
df_estacao.loc[:, "UF"] = uf
    
#df_estacao.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes_teste.csv', index=False)

53


In [276]:
df_estacao['POLUENTE']

0                                 PM2,5
1                                  PM10
2                             MP10,MP25
3                                  PM10
4                                  PM10
5                           MP2,5, MP10
6                           MP2,5, MP10
7                           MP2,5, MP10
8    MP10,NO,CO,PTS,O3,SO2,NO2,NOX,MP25
Name: POLUENTE, dtype: object

In [262]:
def pol_to_station(df_ids):

    base_path = Path('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/')

    id_to_poluentes = {}
    
    for poluente_dir in base_path.iterdir():
        if poluente_dir.is_dir():
            poluente = poluente_dir.name
    
            arquivos = [arq.stem for arq in poluente_dir.glob("*")]
    
            for id_mma in df_ids["ID_MMA"]:
                if any(str(arq).startswith(id_mma) for arq in arquivos):
                    id_to_poluentes.setdefault(id_mma, []).append(poluente)
    
    df_ids["POLUENTE"] = df_ids["ID_MMA"].map(id_to_poluentes).fillna("").apply(lambda x: ",".join(x) if isinstance(x, list) else "")
    
    return(df_ids)

# Varrer MQAr

In [283]:
caminho_MQAr = os.getcwd() + '/data/MQAr/'

lista_pols = os.listdir(caminho_MQAr)

arquivos_MQAr = pd.DataFrame({'ID_MMA_COMPLETO':[]})

for pol in lista_pols:

    if '.ipynb' not in pol and pol != 'METEREOLOGICO':

        caminho_pol = caminho_MQAr+pol

        estacoes_pols = os.listdir(caminho_pol)

        print(pol)

        for estacao in estacoes_pols:

            if estacao != '.ipynb_checkpoints':

                arquivos_MQAr.loc[len(arquivos_MQAr)] = estacao[:-4]

arquivos_MQAr['ID_MMA'] = arquivos_MQAr['ID_MMA_COMPLETO'].str[:6]

MP10
NO
FORMAL
HCT
BENZENO
CO
H2S
ACETAL
PTS
CH4
MP1
O3
ETILBENZENO
SO2
NO2
TOLUENO
VOC
CHUMBO
NH3
OXILENO
HCNM
NOX
ERT
MPXILENO
MP25
XILENO


In [284]:
Monitoramento_MQAr = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')

df_monitoramento = pd.DataFrame({'ID_MMA_COMPLETO':Monitoramento_MQAr['ID_MMA_COMPLETO']})

df_monitoramento['ID_MMA'] = df_monitoramento['ID_MMA_COMPLETO'].str[:6]

arquivos_MQAr['BANCO_DADOS'] = arquivos_MQAr['ID_MMA'].isin(df_monitoramento['ID_MMA']).map({True: 'Sim', False: 'Nao'})

print(arquivos_MQAr)


     ID_MMA_COMPLETO  ID_MMA BANCO_DADOS
0        SP0248RA001  SP0248         Sim
1        RJ0031RA001  RJ0031         Sim
2        RJ0282RA001  RJ0282         Sim
3        SP0085RA001  SP0085         Sim
4        RJ0048RA001  RJ0048         Sim
...              ...     ...         ...
2403     RJ0218RA023  RJ0218         Sim
2404     RJ0012RA023  RJ0012         Sim
2405     RJ0036RA023  RJ0036         Sim
2406     RJ0071RA023  RJ0071         Sim
2407     RJ0216RA023  RJ0216         Sim

[2408 rows x 3 columns]


In [285]:
df_monitoramento

,ID_MMA_COMPLETO,ID_MMA
0,AC0030RA002,AC0030
1,AC0001RA002,AC0001
2,AC0002RA002,AC0002
3,AC0003RA002,AC0003
4,AC0004RA002,AC0004
...,...,...
2361,SP0323RA002,SP0323
2362,TO0003RA002,TO0003
2363,TO0001RA002,TO0001
2364,TO0002RA002,TO0002


In [286]:
num_sim = (arquivos_MQAr['BANCO_DADOS'] == 'Sim').sum()
print(num_sim)



2311


In [288]:
df_nao = arquivos_MQAr[arquivos_MQAr['BANCO_DADOS'] == 'Nao']
print(df_nao)

for id_mma_comp in df_nao['ID_MMA_COMPLETO']:
    if 'SP' in id_mma_comp:
        print(id_mma_comp)

     ID_MMA_COMPLETO  ID_MMA BANCO_DADOS
10       MA1002ND001  MA1002         Nao
13       RJ1014ND001  RJ1014         Nao
16       MA1006ND001  MA1006         Nao
20       RJ1004ND001  RJ1004         Nao
38       SP0084RA001  SP0084         Nao
...              ...     ...         ...
2324     RJ1004ND002  RJ1004         Nao
2331     PA1003ND002  PA1003         Nao
2343     PA1002ND002  PA1002         Nao
2349     MA1004ND002  MA1004         Nao
2351     RN1001ND002  RN1001         Nao

[97 rows x 3 columns]
SP0084RA001
SP0236RA001
SP0084RA017
SP0084RA007
SP0084RA005
SP0236RA005
SP0084RA004
SP0084RA018
